# 🎰 18 · Random variables and their distributions
### Random variables as functions on Ω · events such as {X = x} · the cumulative distribution function · probability mass functions and the point mass · Bernoulli, multinoulli, binomial, Poisson (Bortkiewicz's horse kicks) and geometric · probability density functions: uniform, Gaussian, exponential · PMF vs PDF vs CDF · sampling · read counts, sequencing errors and expression thresholds

**BiologyToMath** · Part E — Probability · Module 18 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/18_random_variables_distributions.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.integrate.quad`, `scipy.special.erf` — to check our formulas), pandas, Matplotlib, `fractions.Fraction` |
| **Prerequisites** | Modules 16–17 (probability spaces, events, independence, conditional probability); Module 14 (derivatives, integrals as areas, $e^x$ and $\ln$); Module 03 (functions, domain, codomain, preimage). |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Random variables: connecting sample spaces, events and data; the definition X: Ω → ℝ; notation X, x, P(X = x), P(X); the two-coin and three-coin examples; a random variable is not an event; several random variables on one sample space; discrete vs continuous random variables; the cumulative distribution function and its properties; probability mass functions; the point-mass distribution; the sum of two 4-sided dice; Bernoulli (with mean and variance), multinoulli, binomial, Poisson (Bortkiewicz's horse kicks) and geometric distributions; probability density functions; the uniform distribution (incl. Uniform(0, 1)); discrete vs continuous uniform; the Gaussian (standard normal, a 2-D Gaussian); the exponential distribution (waiting times between Poisson events); PMF vs PDF (singleton vs small interval); CDFs of discrete and continuous random variables; PMF vs PDF vs CDF; applications: RNA-seq read counts (Poisson), sequencing errors (binomial), CDF thresholds for gene expression |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. Coins, dice, genotypes, sequencing reads, single-cell counts (toolkit `simulate_cell_counts`), mutation waiting times and expression values are **SIMULATED**. Bortkiewicz's horse-kick table (deaths per corps and year: 0, 1, 2, 3, 4 deaths observed in 109, 65, 22, 3, 1 of 200 corps-years) is a **classic published table, typed in**. Base compositions, mutation probabilities, error rates and the expression threshold are **illustrative numbers**.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** a random variable as a function $X: \Omega \to \mathbb{R}$, **compute** its distribution by adding the probabilities of the outcomes that map to each value, and **explain** why a random variable is neither a probability nor an event.
2. **Construct** and **plot** the cumulative distribution function (CDF) of a random variable and **use** its properties ($P(a < X \le b) = F(b) - F(a)$, jumps = point probabilities).
3. **Write** the probability mass functions (PMFs) of the point mass, Bernoulli, multinoulli, binomial, Poisson and geometric distributions, **derive** the binomial PMF by counting sequences, and **derive** the Bernoulli mean $\alpha$ and variance $\alpha(1-\alpha)$.
4. **Fit** a Poisson distribution to Bortkiewicz's horse-kick data and **show** numerically that the binomial converges to the Poisson for many trials with small success probability.
5. **Define** a probability density function (PDF), **compute** probabilities as areas by numerical integration, and **explain** why $f(x)$ can exceed 1 while $P(X = x) = 0$.
6. **Work** with the uniform, Gaussian and exponential distributions: PDFs, CDFs, z-scores, the 68–95–99.7 rule and the link between Poisson counts and exponential waiting times.
7. **Choose** a distribution for a biological quantity (read counts, sequencing errors, waiting times, expression levels) and **compute** tail probabilities such as $P(X > t) = 1 - F(t)$.
8. **Sample** from every distribution with `rng` and **compare** histograms with the PMF or PDF.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports and course helpers | 5 min |
| 1 · From outcomes to numbers | random variables on coins, dice and crosses; events $\{X = x\}$ | 18 min |
| 2 · The cumulative distribution function | step CDFs and their properties | 12 min |
| 3 · Discrete random variables and PMFs | PMF, point mass, the sum of two 4-sided dice | 15 min |
| 4 · Bernoulli and multinoulli | one yes/no trial; one base of DNA; mean and variance | 15 min |
| 5 · Binomial | counting sequences; genotypes; sequencing errors and Phred scores | 20 min |
| 6 · Poisson | rare events; horse kicks; the binomial limit; RNA-seq counts | 22 min |
| 7 · Geometric | waiting for the first success; memorylessness | 12 min |
| 8 · Continuous random variables and PDFs | area = probability; the uniform distribution | 18 min |
| 9 · The Gaussian | z-scores, Φ and erf, 68–95–99.7, 2-D preview, expression thresholds | 20 min |
| 10 · Exponential, summary and sampling | waiting times between Poisson events; PMF vs PDF vs CDF; simulation | 18 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A sequencer reads the same position of your genome 10 times. If you are heterozygous there, how likely is it that
by bad luck only one of the 10 reads shows your second allele, so that the variant is missed? An RNA-seq experiment
counts 12 reads on a gene that usually gets 5: surprising or not? A lineage of bacteria waits for a resistance
mutation: how many generations will it take? A clinical lab flags samples whose marker-gene expression exceeds a
threshold: how often will a healthy sample be flagged?

Each question is about a **number** produced by a random experiment — a count of reads, of molecules, of generations;
an expression level. Probability turns such numbers into **random variables**, and a handful of named
**distributions** — binomial, Poisson, geometric, Gaussian, exponential — describe almost all of them. This module
builds that toolbox, and every later statistical method (estimation, testing, regression) starts from it.

| Biological question | Mathematical tool | Where |
|---|---|---|
| How many heads, mutant alleles, affected children? | a random variable as a function on Ω; its PMF | §1, §3 |
| What is the probability of *at most* $x$? | the cumulative distribution function | §2 |
| Is a base call right or wrong? Which base is it? | Bernoulli, multinoulli | §4 |
| How many of $n$ reads are correct? How many alleles does a person carry? | binomial | §5 |
| How many reads, mutations, horse-kick deaths per unit? | Poisson | §6 |
| How many generations until the first mutation? | geometric | §7 |
| What is a probability for a continuous measurement? | PDF, area under a curve; uniform | §8 |
| How unusual is an expression level? | Gaussian, z-scores, $1 - F(t)$ | §9 |
| How long between rare events? | exponential | §10 |

> 🧪 **Simulated data, with a known truth.** Apart from Bortkiewicz's classic horse-kick table, every data set is
> simulated with a fixed seed, so each formula can be compared with "experiments" whose true distribution we know.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools
from fractions import Fraction
from math import comb, factorial
from scipy import stats
from scipy.integrate import quad
from scipy.special import erf

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_cell_counts` with `CELL_TYPE_MARKERS` produces the single-cell RNA counts of Section 6, and `simulate_genotypes` the Hardy–Weinberg genotypes of Section 5. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd


CELL_TYPE_MARKERS = {
    "T cell": ["CD3E", "CD3D", "IL7R"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "macrophage": ["CD68", "LYZ", "CSF1R"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN"],
}


def simulate_cell_counts(n_cells=(160, 120, 90, 70), cell_types=("T cell", "B cell", "macrophage", "fibroblast"),
                         n_background: int = 28, overdispersion: float = 0.0, seed: int = 680) -> dict:
    """SIMULATED single-cell RNA counts (molecules of each gene detected in each cell).

    Each cell type has a true mean count λ for every gene: its three ``CELL_TYPE_MARKERS`` genes are
    high (λ ≈ 6–15) in that type and low (≈ 0.1–0.5) elsewhere; ``n_background`` other genes share
    similar λ (≈ 0.2–4) in all types.  Counts are Poisson(λ); with ``overdispersion`` φ > 0 each
    cell's λ is first multiplied by a Gamma(1/φ, φ) factor (mean 1), which makes counts negative-
    binomial with variance λ + φλ² — closer to real data.

    Returns a dict with ``counts`` (cells × genes, int), ``labels`` (cell type of each cell),
    ``genes``, ``rates`` (DataFrame cell types × genes of true λ) and ``source``.
    """
    rng = np.random.default_rng(seed)
    markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
    genes = markers + [f"bg{i:02d}" for i in range(1, n_background + 1)]
    rates = np.zeros((len(cell_types), len(genes)))
    bg = rng.uniform(0.2, 4.0, n_background)
    for t, ctype in enumerate(cell_types):
        for j, g in enumerate(markers):
            rates[t, j] = rng.uniform(6.0, 15.0) if g in CELL_TYPE_MARKERS[ctype] else rng.uniform(0.1, 0.5)
        rates[t, len(markers):] = bg * rng.uniform(0.8, 1.25, n_background)
    labels, counts = [], []
    for t, (ctype, n) in enumerate(zip(cell_types, n_cells)):
        lam = np.tile(rates[t], (n, 1))
        if overdispersion > 0:
            lam = lam * rng.gamma(1.0 / overdispersion, overdispersion, size=lam.shape)
        counts.append(rng.poisson(lam))
        labels += [ctype] * n
    return {"counts": np.vstack(counts), "labels": np.array(labels), "genes": genes,
            "rates": pd.DataFrame(rates, index=list(cell_types), columns=genes),
            "source": "SIMULATED single-cell counts (BiologyToMath simulate_cell_counts)"}


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())

print("🧰 toolkit loaded:", ", ".join(sorted(['CELL_TYPE_MARKERS', 'simulate_cell_counts', 'simulate_genotypes'])))

---
## 1 · From outcomes to numbers: random variables

> 📍 **Where we are.** Modules 16–17 worked with sample spaces and events. Statistics, however, deals with **data** —
> numbers. How do we connect sample spaces, events and data? Through random variables, which become the main focus
> from now on.

In probability we are often not interested in the raw outcomes but in **numerical summaries** of them. Toss two
coins: the sample space is $\Omega = \{HH, HT, TH, TT\}$, but the question is *how many heads?* A random variable assigns
numbers to outcomes.

**Definition.** A **random variable** is a function (a mapping) from the sample space to the real numbers,

$$X: \Omega \to \mathbb{R}, \qquad \omega \mapsto X(\omega),$$

*In words:* $X$ takes an outcome $\omega$ and returns a real number $X(\omega)$ — "random" because the outcome is random,
but the rule itself is fixed and deterministic (a function in the sense of Module 03). The mapping is **not a
probability** by itself. (A random variable may also map to vectors, e.g. the expression levels of several genes —
Module 19.)

Random variables are the lens through which we view and analyse data. Once they are introduced, sample spaces fade into
the background — yet behind every random variable lies a sample space, unseen but essential.

**Notation.** Capital $X$ is the random variable; lower-case $x$ is a value it can take (we usually use the lower-case
letter of the same name); $P(X = x)$ is the probability that $X$ takes the value $x$; $P(X)$ denotes the whole
**distribution** of $X$ — the list of all these probabilities. The probability of a value is the sum of the
probabilities of all the outcomes that lead to it:

$$P(X = x) = P\big(\{\omega \in \Omega : X(\omega) = x\}\big) = \sum_{\omega:\, X(\omega) = x} P(\{\omega\}).$$

*In words:* collect the outcomes that $X$ sends to $x$ — the **preimage** of $x$ — and add their probabilities.

In [ ]:
# ── 1.1 Two coin tosses: the random variable C = number of heads, and its distribution ──
Omega2 = ["HH", "HT", "TH", "TT"]
p_omega = {w: Fraction(1, 4) for w in Omega2}                     # fair coins: uniform measure


def count_heads(outcome: str) -> int:
    """The random variable C: Ω → ℝ, C(ω) = number of heads in the outcome ω."""
    return outcome.count("H")


def distribution_of(rv, p_omega: dict) -> dict:
    """P(X = x) for every value x: add p(ω) over all outcomes ω with X(ω) = x (the preimage of x)."""
    pmf = {}
    for w, p in p_omega.items():
        pmf[rv(w)] = pmf.get(rv(w), Fraction(0)) + p
    return dict(sorted(pmf.items()))


print(pd.DataFrame({"ω": Omega2, "p(ω)": [str(p_omega[w]) for w in Omega2],
                    "C(ω)": [count_heads(w) for w in Omega2]}).to_string(index=False))
pmf_C = distribution_of(count_heads, p_omega)
print("\nc       :", list(pmf_C))
print("P(C = c):", [str(p) for p in pmf_C.values()], "  sum =", sum(pmf_C.values()))
print("the EVENT {C = 1} = {ω : C(ω) = 1} =", {w for w in Omega2 if count_heads(w) == 1})

🔍 **What just happened?** The function `count_heads` *is* the random variable $C$: it maps $HH \mapsto 2$, $HT \mapsto
1$, $TH \mapsto 1$, $TT \mapsto 0$. `distribution_of` adds the probabilities of the outcomes with the same value, giving
$P(C = 0) = 1/4$, $P(C = 1) = 1/4 + 1/4 = 1/2$, $P(C = 2) = 1/4$, which sum to 1. The last line shows the difference
between three objects that students often mix up:

* the **random variable** $C$ is a *function* ($C(HT) = 1$ is a number, the value at one outcome);
* "$C = 1$" is an **event** — the *set* $\{HT, TH\}$ of outcomes where the function equals 1;
* $P(C = 1) = 1/2$ is a **probability** — a number attached to that event.

⚠️ **A random variable is NOT an event** (and not a probability).

**A bigger example: three consecutive fair coin tosses.** The probability space $(\Omega, \mathcal{A}, P)$ has
$\Omega = \{HHH, HHT, HTH, HTT, THH, THT, TTH, TTT\}$, $\mathcal{A} = \mathcal{P}(\Omega)$ (all subsets) and $P$ given by
counting, e.g. $P(\Omega) = 1$ and $P(\{HHH, TTT\}) = 2/8 = 1/4$. Define $X$ = the number of heads in the **first** toss and
$Y$ = the number of heads in **all three** tosses. They transform $\Omega$ into new, smaller sets of values,
$X: \Omega \to \{0, 1\}$ and $Y: \Omega \to \{0, 1, 2, 3\}$, each with its own distribution.

In [ ]:
# ── 1.2 Three tosses: two random variables on the same probability space ──
Omega3 = ["".join(t) for t in itertools.product("HT", repeat=3)]
p3 = {w: Fraction(1, 8) for w in Omega3}
print(f"|Ω| = {len(Omega3)},  |𝒜| = |P(Ω)| = 2^8 = {2 ** len(Omega3)},  P(Ω) = {sum(p3.values())},  "
      f"P({{HHH, TTT}}) = {p3['HHH'] + p3['TTT']}")
X_first = lambda w: int(w[0] == "H")            # X: number of heads in the first toss
Y_total = lambda w: w.count("H")                # Y: number of heads in all three tosses
print(pd.DataFrame([[X_first(w) for w in Omega3], [Y_total(w) for w in Omega3]],
                   index=["X(ω)", "Y(ω)"], columns=Omega3).to_string())
for name, rv in [("X", X_first), ("Y", Y_total)]:
    print(f"distribution of {name}:", {x: str(p) for x, p in distribution_of(rv, p3).items()})

🔍 **What just happened?** The probability space has 8 outcomes and $2^8 = 256$ events. Each column of the table is one
outcome, with the values of both random variables: e.g. $THH \mapsto X = 0, Y = 2$. The two distributions are
$P(X = 0) = P(X = 1) = 1/2$ and $P(Y = 0) = 1/8$, $P(Y = 1) = 3/8$ (HTT, THT, TTH), $P(Y = 2) = 3/8$, $P(Y = 3) = 1/8$.

### 1.1 Several random variables on one sample space

One sample space can support many different random variables. Rolling a die, $\Omega = \{1, \ldots, 6\}$: let
$A(\omega)$ = the value of the die and $B(\omega) = 1$ if the value is even, 0 otherwise. In genetics, the cross
$Aa \times Aa$ of Module 16 supports the **genotype** $G$ = number of $a$ alleles the child receives and the
**disease indicator** $D = 1$ if the child is $aa$ (recessive phenotype). This idea prepares the joint distributions
and multivariate models of [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb).

In [ ]:
# ── 1.3 Several random variables on one Ω: a die, and a Mendelian cross ──
die = {w: Fraction(1, 6) for w in range(1, 7)}
value = lambda w: w
is_even = lambda w: int(w % 2 == 0)
print("die:  ω    ", list(die), "\n      A(ω) ", [value(w) for w in die], "\n      B(ω) ", [is_even(w) for w in die])
print("      distribution of B:", {b: str(p) for b, p in distribution_of(is_even, die).items()})

cross = {(m, f): Fraction(1, 4) for m in "Aa" for f in "Aa"}       # (mother's allele, father's allele)
G = lambda w: (w[0] == "a") + (w[1] == "a")                         # number of a alleles
D = lambda w: int(G(w) == 2)                                        # 1 = affected (recessive)
print("\ncross: G:", {g: str(p) for g, p in distribution_of(G, cross).items()},
      "   D:", {d: str(p) for d, p in distribution_of(D, cross).items()})

🔍 **What just happened?** On the die, $A$ simply returns the face and $B$ compresses it to "even or not", with
$P(B = 0) = P(B = 1) = 1/2$. On the cross, the same four equally likely outcomes give the genotype distribution
$P(G = 0) = 1/4$, $P(G = 1) = 1/2$, $P(G = 2) = 1/4$ and the disease distribution $P(D = 1) = 1/4$, $P(D = 0) = 3/4$.
$D$ is a function of $G$, so the two random variables are strongly related — exactly the kind of relationship that
joint distributions describe in [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb).

### 1.2 Discrete and continuous random variables

| | **discrete** random variable | **continuous** random variable |
|---|---|---|
| takes … | separate (countably many) values | any value in an interval |
| biological examples | coin flips, nucleotides in genomes, disease state, RNA-seq read counts, number of offspring | height, weight, blood pressure, cholesterol level, expression level, waiting time |
| described by | a **probability mass function** (PMF) | a **probability density function** (PDF) |
| both have | a **cumulative distribution function** (CDF) | |

### 📘 Lecture notes — random variables

**Why:** probability starts from sample spaces and events; data are numbers. A random variable is the bridge — a
mapping from the sample space to real values (or vectors). Once random variables are introduced they become the main
focus; the sample space is always there in the background.

**Definition:** $X: \Omega \to \mathbb{R}$ assigns a real number $X(\omega)$ to each outcome $\omega$; it is **not a
probability by itself**. **Notation:** $X$ random variable; $x$ a value; $P(X = x)$ probability of $X = x$; $P(X)$ the
distribution of $X$. $P(X = x)$ = sum of the probabilities of all outcomes that lead to $x$.

*Two coins:* $C$ = number of heads: $HH \mapsto 2$, $HT \mapsto 1$, $TH \mapsto 1$, $TT \mapsto 0$ (each $p(\omega) =
1/4$) ⇒ $P(C = 0) = 1/4$, $P(C = 1) = 1/2$, $P(C = 2) = 1/4$. *Three coins:* $(\Omega, \mathcal{P}(\Omega), P)$,
$P(\{HHH, TTT\}) = 2/8$; $X$ = heads in the first toss ($\Omega \to \{0, 1\}$), $Y$ = heads in all three
($\Omega \to \{0, 1, 2, 3\}$).

**Misconception:** a random variable is NOT an event: $X(HT) = 1$ is a value; the event "$X = 1$" is the set $\{HT, TH\}$.

**Several random variables on one $\Omega$:** die value $A(\omega)$ and parity $B(\omega)$ — the basis of joint
distributions. **Discrete** (coin flips, nucleotides, disease state, read counts → PMF) vs **continuous** (height,
weight, blood pressure, cholesterol → PDF).

> 🎓 **Exam tip.** Asked for "the distribution of $X$", draw the table $\omega \mid p(\omega) \mid X(\omega)$ first, then
> group equal values of $X$ and add their probabilities.

✏️ **Your turn.** (1) For three tosses define $Z$ = (number of heads) − (number of tails). What values does $Z$ take,
and what is its distribution? (2) For the cross, define $H = 1$ if the child is a heterozygous carrier. Write the event
$\{H = 1\}$ as a set of outcomes and compute $P(H = 1)$.

---
## 2 · The cumulative distribution function

> 📍 **Where we are.** A distribution can be described by listing $P(X = x)$ for every value — but that only works for
> discrete variables. One description works for *every* random variable: the cumulative distribution function.

**Definition.** Given a random variable $X$, its **cumulative distribution function** (CDF, or simply *distribution
function*) is the function $F_X: \mathbb{R} \to [0, 1]$ defined by

$$F_X(x) = P(X \le x).$$

*In words:* for every real number $x$ — not only for the values $X$ can take — $F_X(x)$ is the probability that $X$
comes out at most $x$. It accumulates the probability from $-\infty$ up to $x$.

**Example.** Flip a fair coin twice and let $X$ be the number of heads. Then $P(X = 0) = P(X = 2) = \tfrac14$ and
$P(X = 1) = \tfrac12$, and the distribution function is

$$F_X(x) = \begin{cases} 0 & x < 0\\ 1/4 & 0 \le x < 1\\ 3/4 & 1 \le x < 2\\ 1 & x \ge 2.\end{cases}$$

*In words:* below 0 nothing has accumulated; from 0 on we have the probability of 0 heads (1/4); from 1 on also that of 1
head (1/4 + 1/2 = 3/4); from 2 on everything (1). The CDF effectively contains **all** the information about the random
variable: the size of each jump is the probability of the value where it occurs.

In [ ]:
# ── 2.1 The step CDF of the number of heads in two tosses ──
def cdf_from_pmf(pmf: dict):
    """F(x) = P(X ≤ x) = sum of P(X = v) over the values v ≤ x (works for every real x)."""
    return lambda x: sum((p for v, p in pmf.items() if v <= x), Fraction(0))


def plot_step_cdf(ax, pmf: dict, xlim: tuple, colour: str = "k"):
    """Draw a discrete CDF: flat pieces, a filled dot where each value is included, an open dot just before it."""
    F = cdf_from_pmf(pmf)
    edges = [xlim[0]] + list(pmf) + [xlim[1]]
    for a, b in zip(edges[:-1], edges[1:]):
        ax.plot([a, b], [float(F(a)), float(F(a))], color=colour, lw=2.5)
    for v in pmf:
        ax.plot(v, float(F(v)), "o", color=colour, ms=6)                                    # F(v) includes P(X = v)
        ax.plot(v, float(F(v) - pmf[v]), "o", mfc="white", mec=colour, ms=6)                # value just below v
    ax.set_xlim(*xlim); ax.set_ylim(-0.05, 1.08); ax.set_ylabel("F(x) = P(X ≤ x)")


F_C = cdf_from_pmf(pmf_C)
for x in [-1, 0, 0.5, 1, 1.5, 2, 3.7]:
    print(f"F({x:>4}) = {F_C(x)}")
fig, ax = plt.subplots(figsize=(6, 3.2))
plot_step_cdf(ax, pmf_C, (-1.5, 3.5))
ax.set_xlabel("x (number of heads)"); ax.set_title("CDF of the number of heads in two fair tosses")
plt.tight_layout(); plt.show()
grid = np.linspace(-3, 5, 801)
values_F = np.array([float(F_C(x)) for x in grid])
print("non-decreasing:", bool(np.all(np.diff(values_F) >= 0)), "| F(−3) =", values_F[0], "| F(5) =", values_F[-1])
eps = 1e-9
print(f"at x = 1: F(1 − ε) = {F_C(1 - eps)}, F(1) = {F_C(1)}, F(1 + ε) = {F_C(1 + eps)}  → jump {F_C(1) - F_C(1 - eps)} = P(X = 1)")
print("P(0 < X ≤ 2) = F(2) − F(0) =", F_C(2) - F_C(0))

🔍 **What just happened?** `cdf_from_pmf` returns a function that works for *every* real input: $F(-1) = 0$,
$F(0) = F(0.5) = 1/4$, $F(1) = F(1.5) = 3/4$, $F(2) = F(3.7) = 1$. The plot is a **step function** going up to 1: flat
between the possible values, with a jump at each one. The filled dots mark that the value itself is included
($F(1) = 3/4$, because "$\le$"), the open dots that the limit from the left is lower ($1/4$). The numerical checks
confirm the general properties:

* $F$ is **non-decreasing** (accumulating probability can only add);
* $\lim_{x \to -\infty} F(x) = 0$ and $\lim_{x \to \infty} F(x) = 1$;
* $F$ is **right-continuous**: $F(1 + \varepsilon) = F(1)$, while from the left it jumps, and the jump size
  $F(1) - F(1^-) = 1/2$ is exactly $P(X = 1)$;
* probabilities of intervals are differences: $P(a < X \le b) = F(b) - F(a)$, e.g. $P(0 < X \le 2) = 1 - 1/4 = 3/4$.

✏️ **Your turn.** (1) Plot the CDF of $Y$ (heads in three tosses) with `plot_step_cdf`. Read off $P(Y \le 1)$ and
$P(1 < Y \le 3)$. (2) Explain why a CDF can never decrease, using monotonicity from Module 16 ($\{X \le a\} \subseteq
\{X \le b\}$ when $a \le b$).

---
## 3 · Discrete random variables and probability mass functions

> 📍 **Where we are.** For a variable with separate values, listing the probability of each value is the most direct
> description. That list, viewed as a function, is the probability mass function.

**Definition.** $X$ is **discrete** if it takes countably many values $\{x_1, x_2, \ldots\}$. Its **probability
function** or **probability mass function** (PMF) is

$$f_X(x) = P(X = x), \qquad f_X(x) \ge 0 \text{ for all } x \in \mathbb{R}, \qquad \sum_i f_X(x_i) = 1.$$

*In words:* the PMF gives the probability of each value; it is never negative, it is 0 at every $x$ that $X$ cannot
take, and its values add up to 1 (Axiom 2). Sometimes we write $f$ instead of $f_X$, or $p(x)$. The CDF is related to
the PMF by

$$F_X(x) = P(X \le x) = \sum_{x_i \le x} f_X(x_i).$$

*In words:* the CDF at $x$ adds up the PMF over all *values* $x_i$ of $X$ that do not exceed $x$. ⚠️ The sum runs over
the **values** in the range of $X$, not over the outcomes $\omega$ in $\Omega$ (outcomes such as "HT" cannot be
compared with a number $x$; their values $X(\omega)$ can).

In [ ]:
# ── 3.1 The PMF of C (two tosses) as a picture ──
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
c_values = list(pmf_C); c_probs = [float(p) for p in pmf_C.values()]
axes[0].plot(c_values, c_probs, "o", ms=9, mfc="none", mec="tab:blue", mew=1.5)
axes[0].set_xticks(c_values); axes[0].set_xlim(-0.5, 2.5); axes[0].set_ylim(0, 1)
axes[0].set_xlabel("c"); axes[0].set_ylabel("p(C = c)"); axes[0].set_title("PMF of C: points at the values")
axes[1].vlines(c_values, 0, c_probs, color="tab:blue", lw=2); axes[1].plot(c_values, c_probs, "o", color="tab:blue")
axes[1].set_xticks(c_values); axes[1].set_xlim(-0.5, 2.5); axes[1].set_ylim(0, 1)
axes[1].set_xlabel("c"); axes[1].set_title("the same PMF as a 'stem' plot")
plt.tight_layout(); plt.show()
print("f(0.5) =", pmf_C.get(0.5, 0), "(not a value of C)  |  Σ f(c) =", sum(pmf_C.values()))

🔍 **What just happened?** A PMF is drawn as isolated points (left) or "stems" (right) at the values 0, 1, 2 with heights
1/4, 1/2, 1/4. Between the values the PMF is 0 — $f(0.5) = 0$ because two tosses cannot give half a head — and the
heights add up to 1. Compare with the CDF of Section 2: the PMF shows *where* the probability sits, the CDF how much has
accumulated.

### 3.1 The point-mass distribution

The simplest distribution of all puts all its probability on one value $a$. $X$ has a **point mass distribution** at
$a$, written $X \sim \delta_a$, if $P(X = a) = 1$. Then the PMF is $f(x) = 1$ for $x = a$ and 0 otherwise, and

$$F(x) = \begin{cases} 0 & x < a\\ 1 & x \ge a. \end{cases}$$

*In words:* the CDF is 0 until $a$ and jumps to 1 there — a "random" variable without any randomness. Biology: in a fully
**inbred** mouse strain every animal is homozygous at (almost) every locus, so the number of copies of the reference
allele is $\delta_2$; a measurement of a quantity that never varies is a point mass.

In [ ]:
# ── 3.2 The point mass δ_2 (an inbred, homozygous strain: 2 copies of the allele, always) ──
point_mass = {2: Fraction(1)}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.0))
axes[0].vlines([2], 0, [1], color="tab:purple", lw=2); axes[0].plot([2], [1], "o", color="tab:purple")
axes[0].set_xlim(-0.5, 3.5); axes[0].set_ylim(0, 1.1); axes[0].set_xticks([0, 1, 2, 3])
axes[0].set_xlabel("x (copies of the allele)"); axes[0].set_ylabel("f(x)"); axes[0].set_title("PMF of δ₂")
plot_step_cdf(axes[1], point_mass, (-0.5, 3.5), colour="tab:purple")
axes[1].set_xlabel("x"); axes[1].set_title("CDF of δ₂: a single jump of size 1 at x = 2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The PMF of $\delta_2$ is a single stem of height 1 at $x = 2$; its CDF is 0 up to 2 and
jumps to 1 at 2 (filled dot: $F(2) = 1$). Every other distribution in this module spreads the probability over more
than one value.

### 3.2 The sum of two 4-sided dice

As another example, roll two 4-sided dice (simpler than 6-sided ones) and let $V(\omega)$ be the sum of the two values.
All 16 ordered outcomes $\omega = 11, 12, \ldots, 44$ have probability 1/16; the random variable $V$ maps them to the
values 2 to 8.

In [ ]:
# ── 3.3 The sum V of two 4-sided dice: 16 outcomes, the PMF and the CDF ──
dice = {(i, j): Fraction(1, 16) for i in range(1, 5) for j in range(1, 5)}
V = lambda w: w[0] + w[1]
grid_sums = pd.DataFrame([[i + j for j in range(1, 5)] for i in range(1, 5)],
                         index=[f"die 1 = {i}" for i in range(1, 5)], columns=[f"die 2 = {j}" for j in range(1, 5)])
print("V(ω) for the 16 equally likely outcomes:\n", grid_sums.to_string())
pmf_V = distribution_of(V, dice)
print("\nv        :", list(pmf_V))
print("p(V = v)  :", [str(p) for p in pmf_V.values()])
F_V = cdf_from_pmf(pmf_V)
print("\np(V ≤ 4) = p(V = 2) + p(V = 3) + p(V = 4) =", " + ".join(str(pmf_V[v]) for v in (2, 3, 4)), "=", F_V(4))
print("p(V ≤ 4.78) =", F_V(4.78), "  (the CDF is defined between the values, too)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(list(pmf_V), [float(p) for p in pmf_V.values()], "o", ms=8, mfc="none", mew=1.5)
axes[0].set_ylim(0, 1); axes[0].set_xlabel("v (sum of the two dice)"); axes[0].set_ylabel("p(V = v)")
axes[0].set_title("PMF: 1/16, 1/8, 3/16, 1/4, 3/16, 1/8, 1/16")
plot_step_cdf(axes[1], pmf_V, (0, 10), colour="tab:red")
axes[1].axvline(4.78, color="0.5", ls=":", lw=1); axes[1].set_xlabel("v"); axes[1].set_title("CDF: P(V ≤ 4) = P(V ≤ 4.78) = 3/8")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 4 × 4 table lists the sum for every ordered pair of faces; counting equal sums along the
anti-diagonals gives the PMF $\tfrac1{16}, \tfrac18, \tfrac3{16}, \tfrac14, \tfrac3{16}, \tfrac18, \tfrac1{16}$ for
$v = 2, \ldots, 8$ — a symmetric "tent" peaking at 5 (4 of the 16 outcomes). The CDF accumulates it:
$P(V \le 4) = \tfrac1{16} + \tfrac18 + \tfrac3{16} = \tfrac38$, and because nothing happens between 4 and 5,
$P(V \le 4.78) = \tfrac38$ as well. The red staircase climbs $\tfrac1{16}, \tfrac3{16}, \tfrac6{16}, \tfrac{10}{16},
\tfrac{13}{16}, \tfrac{15}{16}, 1$.

### 📘 Lecture notes — the CDF and the PMF

**CDF:** $F_X: \mathbb{R} \to [0, 1]$, $F_X(x) = P(X \le x)$ — for discrete and continuous random variables. Properties:
non-decreasing, right-continuous, $F(-\infty) = 0$, $F(\infty) = 1$; $P(a < X \le b) = F(b) - F(a)$; the CDF
effectively contains all the information about the random variable. *Two fair coins, $X$ = heads:* $F = 0$ ($x < 0$),
$1/4$ ($0 \le x < 1$), $3/4$ ($1 \le x < 2$), $1$ ($x \ge 2$) — a step function up to 1, defined for all $x$.

**PMF** (discrete $X$, countably many values $x_1, x_2, \ldots$): $f_X(x) = P(X = x)$; $f_X(x) \ge 0$ for all $x$,
$\sum_i f_X(x_i) = 1$; $F_X(x) = \sum_{x_i \le x} f_X(x_i)$ (sum over values of $X$). *Two coins:* $f_X = 1/4, 1/2, 1/4$
at $0, 1, 2$, and 0 otherwise.

**Point mass** $X \sim \delta_a$: $P(X = a) = 1$; $f(a) = 1$, $f(x) = 0$ otherwise; $F(x) = 0$ for $x < a$, 1 for $x \ge a$.

**Sum of two 4-sided dice** $V$: PMF $1/16, 1/8, 3/16, 1/4, 3/16, 1/8, 1/16$ on $2, \ldots, 8$;
$P(V \le 4) = \sum_{d \in \{2,3,4\}} P(V = d) = 1/16 + 1/8 + 3/16 = 3/8$; and $P(V \le 4.78) = 3/8$.

> 🎓 **Exam tip.** A discrete CDF is a staircase: jumps at the values, of height $P(X = x)$, filled dot on top. Make
> sure you can go both ways — PMF → CDF by accumulating, CDF → PMF by reading the jump sizes.

✏️ **Your turn.** (1) Compute the PMF of the **maximum** of two 4-sided dice with `distribution_of`. Which value is most
likely? (2) From the staircase of $V$, read off $P(3 < V \le 6)$ as a difference of two CDF values and check it against
the PMF.

---
## 4 · Bernoulli and multinoulli: one trial

> 📍 **Where we are.** From now on we meet the **well-known PMFs** — families of distributions that appear over and
> over, each indexed by a few **parameters**. The simplest describes a single yes/no trial.

**Bernoulli distribution.** A random variable with $\Omega = \{0, 1\}$, "success" $k = 1$ with probability
$0 \le \alpha \le 1$:

$$p(k) = \begin{cases} \alpha & k = 1\\ 1 - \alpha & k = 0 \end{cases} \qquad\text{or, put another way,}\qquad p(k) = \alpha^{k}(1 - \alpha)^{1-k} \quad \forall k \in \Omega.$$

*In words:* the second form packs both cases into one formula — for $k = 1$ it gives $\alpha^1 (1-\alpha)^0 = \alpha$,
for $k = 0$ it gives $\alpha^0 (1-\alpha)^1 = 1 - \alpha$ (anything to the power 0 is 1). Example scenario: a single,
possibly biased, coin flip with probability of heads $\alpha$. Biology: is a randomly chosen person a carrier? Is a base
call correct? Is a gene detected in a cell?

**Mean and variance (a preview of [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb)).** The **expectation** of a discrete random variable is the
probability-weighted average of its values, $E[X] = \sum_i x_i\, p(x_i)$, and its **variance** is
$V(X) = E[X^2] - (E[X])^2$. For a Bernoulli variable, $P(X = 0) = 1 - \alpha$ and $P(X = 1) = \alpha$, so

$$\begin{aligned}
E[X] &= 1 \cdot P(X = 1) + 0 \cdot P(X = 0) = \alpha,\\
E[X^2] &= 1^2 \cdot P(X = 1) + 0^2 \cdot P(X = 0) = \alpha,\\
V(X) &= E[X^2] - (E[X])^2 = \alpha - \alpha^2 = \alpha(1 - \alpha).
\end{aligned}$$

*In words:* the average of a 0/1 variable is the probability of a 1. To get $E[X^2]$ we square the **values** (0 and 1,
which do not change when squared) and weight them by the *unchanged* probabilities. ⚠️ A common slip is to square the
probabilities as well — it is $1^2 \cdot P(X = 1)$, not $1^2 \cdot P(X = 1)^2$. The variance $\alpha(1 - \alpha)$ is
largest for a fair coin ($\alpha = 1/2$) and zero when the outcome is certain ($\alpha = 0$ or 1).

In [ ]:
# ── 4.1 Bernoulli: both forms of the PMF, mean and variance (formula vs SIMULATION) ──
def bernoulli_pmf(k: int, alpha: float) -> float:
    """p(k) = α^k (1 − α)^(1 − k) for k ∈ {0, 1}."""
    return alpha ** k * (1 - alpha) ** (1 - k)


alpha = 0.3
print("p(1) =", bernoulli_pmf(1, alpha), " p(0) =", bernoulli_pmf(0, alpha), " (cases form: α = 0.3, 1 − α = 0.7)")
rng = np.random.default_rng(SEED + 41)
draws = (rng.random(100_000) < alpha).astype(int)
print(f"formula: E[X] = {alpha}, V(X) = α(1 − α) = {alpha * (1 - alpha):.3f}")
print(f"100,000 SIMULATED draws: mean {draws.mean():.4f}, variance {draws.var():.4f}")
alphas = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.plot(alphas, alphas * (1 - alphas), color="tab:green")
ax.set_xlabel("α = P(X = 1)"); ax.set_ylabel("V(X) = α(1 − α)")
ax.set_title("a Bernoulli variable is most variable when α = 1/2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both forms of the PMF give $p(1) = 0.3$ and $p(0) = 0.7$. The formulas give $E[X] = 0.3$ and
$V(X) = 0.21$, and 100,000 simulated draws come within a few thousandths of both. The parabola $\alpha(1 - \alpha)$
peaks at 0.25 for $\alpha = 1/2$: a carrier status with frequency 4 % is far more predictable (variance 0.038) than a
fair coin.

**Multinoulli (categorical) distribution.** A discrete random variable that takes values in $\{1, \ldots, k\}$ with
probabilities $p_1, \ldots, p_k$ (non-negative, summing to 1) has a **multinoulli** distribution,
$X \sim \text{Multinoulli}(p_1, \ldots, p_k)$ — the Bernoulli with more than two outcomes:

$$P(X = x) = p_1^{\delta(x, 1)} \cdot p_2^{\delta(x, 2)} \cdots p_k^{\delta(x, k)}, \qquad \delta(x, i) = \begin{cases} 1 & \text{if } x = i\\ 0 & \text{if } x \ne i. \end{cases}$$

*In words:* $\delta(x, i)$ is an **indicator** (Module 03). For the observed value $x$ exactly one exponent is 1 and all
others are 0, so all factors but one equal 1 and the product is just $p_x$. Biology: the base at a position of a genome
(A, C, G, T coded 1–4), the cell type of a random cell, the genotype at a SNP.

In [ ]:
# ── 4.2 Multinoulli: the base at a position in a GC-rich region (ILLUSTRATIVE composition) ──
bases = ["A", "C", "G", "T"]
p_base = np.array([0.2, 0.3, 0.3, 0.2])                     # p1 … p4, sums to 1


def multinoulli_pmf(x: int, p: np.ndarray) -> float:
    """P(X = x) = Π_i p_i^δ(x, i) with δ(x, i) = 1 if x = i else 0 (values x = 1 … k)."""
    return float(np.prod([p_i ** int(x == i) for i, p_i in enumerate(p, start=1)]))


print("P(X = x) by the product formula:", [round(multinoulli_pmf(x, p_base), 3) for x in range(1, 5)])
rng = np.random.default_rng(SEED + 42)
sample = rng.choice(bases, size=10_000, p=p_base)
freq = pd.Series(sample).value_counts(normalize=True).reindex(bases)
print("10,000 SIMULATED bases:", freq.round(4).to_dict())

🔍 **What just happened?** The product formula returns exactly $p_1, \ldots, p_4 = 0.2, 0.3, 0.3, 0.2$: each time only
one factor survives. Ten thousand simulated bases have frequencies within 0.005 of these probabilities (0.2034, 0.2981, 0.3031,
0.1954). A
multinoulli with $k = 2$ is a Bernoulli; repeating a multinoulli trial $n$ times and counting each category gives the
multinomial distribution (the base composition of a whole read).

✏️ **Your turn.** (1) What is the variance of a Bernoulli variable for a carrier frequency of 1/25? (2) Write the PMF of a
genotype $G \in \{0, 1, 2\}$ with probabilities $(0.49, 0.42, 0.09)$ in multinoulli form (shift the values to 1, 2, 3)
and evaluate it.

---
## 5 · The binomial distribution: counting successes

> 📍 **Where we are.** Repeat a Bernoulli trial $n$ times independently and count the successes. This is perhaps the
> most well-known PMF of all — the number of heads in $n$ flips, of carriers among $n$ patients, of correct reads
> among $n$ reads.

The probability of a (single) trial's success is $0 \le \alpha \le 1$; now we have $n$ independent trials and want the
probability of getting $k \le n$ successes, so $\Omega = \{0, 1, 2, \ldots, n\}$. **Derivation:**

1. Take one *particular* sequence of outcomes with $k$ successes and $n - k$ failures, e.g. $SSF\ldots F$. Because the
   trials are independent, its probability is the product (Module 17) of $k$ factors $\alpha$ and $n - k$ factors
   $1 - \alpha$: $\alpha^k (1 - \alpha)^{n-k}$ — the same for every such sequence, whatever the order.
2. Count the sequences with exactly $k$ successes: we must choose which $k$ of the $n$ positions are successes, which can
   be done in $\binom{n}{k}$ ways.
3. These sequences are disjoint events, so by Axiom 3 we add their probabilities: $\binom{n}{k}$ copies of
   $\alpha^k (1-\alpha)^{n-k}$.

🧮 **Math refresher — factorials and "n choose k".** $n! = n \times (n-1) \times \cdots \times 2 \times 1$ ("$n$
factorial"), with $0! = 1$; e.g. $4! = 24$. The **binomial coefficient** $\binom{n}{k} = \frac{n!}{k!\,(n-k)!}$ ("$n$
choose $k$") counts the subsets of size $k$ of an $n$-element set. Example: $\binom{4}{2} = \frac{24}{2 \cdot 2} = 6$
(the 6 ways to place 2 heads among 4 tosses: HHTT, HTHT, HTTH, THHT, THTH, TTHH). They form Pascal's triangle,
$\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k}$.

$$p(k) = \binom{n}{k}\,\alpha^{k}\,(1 - \alpha)^{n-k} \qquad \forall k \in \Omega = \{0, 1, \ldots, n\}.$$

*In words:* (number of ways to place $k$ successes) × (probability of any one such arrangement). We write
$X \sim \text{Binomial}(n, \alpha)$. A Bernoulli variable is the case $n = 1$.

In [ ]:
# ── 5.1 The binomial PMF by brute-force enumeration of all 2^n sequences (n = 4, α = 0.3) ──
def binomial_pmf(k: int, n: int, alpha: float) -> float:
    """p(k) = C(n, k) α^k (1 − α)^(n − k)."""
    return comb(n, k) * alpha ** k * (1 - alpha) ** (n - k)


n, alpha = 4, 0.3
by_k = {k: 0.0 for k in range(n + 1)}
for seq in itertools.product([1, 0], repeat=n):                    # 1 = success, 0 = failure
    k = sum(seq)
    by_k[k] += alpha ** k * (1 - alpha) ** (n - k)                   # probability of this one sequence
rows = [(k, comb(n, k), round(alpha ** k * (1 - alpha) ** (n - k), 4), round(by_k[k], 4),
         round(binomial_pmf(k, n, alpha), 4), round(stats.binom.pmf(k, n, alpha), 4)) for k in range(n + 1)]
print(pd.DataFrame(rows, columns=["k", "C(4,k) sequences", "prob. of one", "enumeration", "formula", "scipy"]).to_string(index=False))
print("4! =", factorial(4), "| C(4, 2) =", comb(4, 2), "| Pascal: C(4,2) = C(3,1) + C(3,2) =", comb(3, 1) + comb(3, 2))

🔍 **What just happened?** The loop visited all $2^4 = 16$ success/failure sequences and added up the probabilities of
those with the same number of successes. For $k = 2$ there are $\binom42 = 6$ sequences, each with probability
$0.3^2 \times 0.7^2 = 0.0441$, totalling 0.2646 — and the formula and `scipy.stats.binom.pmf` agree for every $k$. The
derivation *is* the computation: enumeration is just too slow for large $n$ ($2^{40} \approx 10^{12}$ sequences), while
the formula is instant.

In [ ]:
# ── 5.2 Binomial PMFs: how the distribution moves with α (n = 40) and with μ (N = 15) ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
k40 = np.arange(0, 41)
for a, colour in [(0.1, "tab:blue"), (0.3, "tab:red"), (0.5, "tab:green")]:
    axes[0].plot(k40, [binomial_pmf(k, 40, a) for k in k40], "o--", ms=4, mfc="none", color=colour, label=f"α = {a}")
axes[0].set_xlabel("k (number of successes in n = 40 trials)"); axes[0].set_ylabel("p(k)")
axes[0].set_title("Binomial(40, α)"); axes[0].legend()
k15 = np.arange(0, 16)
for mu, colour in [(0.1, "tab:blue"), (0.4, "tab:orange"), (0.75, "tab:green")]:
    axes[1].bar(k15, stats.binom.pmf(k15, 15, mu), color=colour, alpha=0.45, label=f"μ = {mu}")
axes[1].set_xlabel("m (number of observations x = 1 in N = 15 experiments)"); axes[1].set_ylabel("p(m)")
axes[1].set_title("Binomial(15, μ) — μ is the success probability"); axes[1].legend()
plt.tight_layout(); plt.show()
for a in [0.1, 0.3, 0.5]:
    pk = np.array([binomial_pmf(k, 40, a) for k in k40])
    print(f"n = 40, α = {a}: most likely k = {k40[pk.argmax()]}, Σ k·p(k) = {np.sum(k40 * pk):.1f} = nα = {40 * a:.1f}")

🔍 **What just happened?** Left: with $\alpha = 0.1$ most outcomes are near 0 (peak at $k = 4$); with $\alpha = 0.3$ the
peak moves right ($k = 12$); with $\alpha = 0.5$ the distribution is symmetric around $n/2 = 20$. Right: the same
family for $N = 15$ trials. ⚠️ **Notation:** many books call the success probability $p$, $\theta$ or — as in this
panel — $\mu$, and the number of trials $N$; here $\mu$ is the probability of success in one trial (e.g. heads in one
coin flip) and $p(m)$ the probability of exactly $m$ successes. It is the same distribution as $\text{Binomial}(n,
\alpha)$ with $n = N$, $\alpha = \mu$. The printed check shows that the **mean** $\sum_k k\,p(k)$ equals $n\alpha$ (4, 12
and 20) — the binomial is a sum of $n$ Bernoulli variables, each with mean $\alpha$ ([Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) proves it).

**Biology: the genotype is binomial.** At a SNP with allele frequency $p$, each of a person's two chromosomes carries the
allele independently with probability $p$ (random mating), so the number of copies is
$G \sim \text{Binomial}(2, p)$ — the Hardy–Weinberg proportions $(1-p)^2, 2p(1-p), p^2$ are just the binomial PMF with
$n = 2$.

In [ ]:
# ── 5.3 Genotypes are Binomial(2, p): SIMULATED population vs the PMF ──
p_allele = 0.3
g = simulate_genotypes(20_000, p_allele, seed=1866)
print("observed frequencies of 0/1/2 copies:", np.round(np.bincount(g, minlength=3) / g.size, 4))
print("Binomial(2, 0.3) PMF:                ", np.round([binomial_pmf(k, 2, p_allele) for k in range(3)], 4))

🔍 **What just happened?** In 20,000 simulated people the genotype frequencies match $\text{Binomial}(2, 0.3) =
(0.49, 0.42, 0.09)$ to two or three decimals. Two copies of a chromosome = two Bernoulli trials.

### 🌉 Bridge — sequencing errors and variant calling with the binomial

In DNA sequencing each read reports a base that is correct with some probability $\alpha$; base-calling software reports
the error probability as a **Phred quality score** $Q = -10\log_{10} P(\text{error})$, so $Q = 20$ means an error rate
of 1 %, $Q = 30$ of 0.1 %. If $n$ reads cover a base and errors are independent, the number of correct reads is
$X \sim \text{Binomial}(n, \alpha)$:

$$P(X = k) = \binom{n}{k}\alpha^k (1 - \alpha)^{n-k}.$$

*In words:* the binomial counts correct reads among $n$ independent reads, exactly as it counts heads. Two questions
that matter for **quality control and variant calling** (Li 2011): (1) how likely is it that a *majority* of reads is
wrong, so that consensus calling fails? (2) At a **heterozygous** site half of the DNA molecules carry each allele, so
the number of reads showing the alternative allele is $\text{Binomial}(n, 1/2)$ — how likely is it to see so few of them
that the variant looks like an error?

In [ ]:
# ── 5.4 Phred scores, majority errors and missed heterozygotes (independent errors assumed) ──
for Q in [10, 20, 30, 40]:
    print(f"Phred Q = {Q}: P(error) = 10^(−Q/10) = {10 ** (-Q / 10):g}")
rows = []
for depth in [1, 3, 5, 10, 20, 30]:
    wrong_majority = {e: stats.binom.sf(depth // 2, depth, e) for e in (0.01, 0.1)}    # P(#wrong reads > depth/2)
    missed_het = stats.binom.cdf(1, depth, 0.5)                                        # ≤ 1 read with the alt allele
    rows.append((depth, f"{wrong_majority[0.01]:.2e}", f"{wrong_majority[0.1]:.2e}", round(missed_het, 4)))
print(pd.DataFrame(rows, columns=["depth n", "P(majority wrong), Q20", "P(majority wrong), Q10",
                                  "P(heterozygote shows ≤ 1 alt read)"]).to_string(index=False))

🔍 **What just happened?** A Phred score of 20 means a 1 % error rate, 30 means 0.1 %. With Q20 reads, a majority of
wrong reads becomes astronomically unlikely as depth grows ($9.9 \times 10^{-6}$ at depth 5, $1.3 \times 10^{-24}$ at
depth 30), and even with poor Q10 reads it is 0.028 at depth 3 but $3.7 \times 10^{-9}$ at depth 30 — the reason sequencing depth
buys accuracy. The heterozygote column is more sobering: at depth 10 a true heterozygote shows at most one alternative
read with probability $11/1024 \approx 0.011$, so about 1 in 100 heterozygous sites at that depth could be dismissed as
an error; at depth 20 the probability is $21/2^{20} = 0.00002$. (Real variant callers also model that an error produces
one of *three* wrong bases and that errors cluster — but the binomial is the starting point.)

✏️ **Your turn.** (1) Use `binomial_pmf` to compute the probability that 100 patients include **no** carrier when the
carrier frequency is 0.04. (2) At what depth does $P(\text{heterozygote shows} \le 1 \text{ alt read})$ first drop below
0.001?

---
## 6 · The Poisson distribution: counting rare events

> 📍 **Where we are.** Many biological counts have no fixed number of "trials": reads on a gene, mutations in a
> chromosome, molecules in a cell, cases of a rare disease per year. The Poisson distribution describes such counts.

Poisson distributions model the probability of a given number of events $k$ occurring in a **fixed interval of time or
space**, when the events happen **independently** and with a **constant average rate**. With $\lambda > 0$ the expected
number of events in the interval:

$$p(k) = \frac{\lambda^k e^{-\lambda}}{k!} \qquad \forall k \in \Omega = \{0, 1, 2, \ldots\}.$$

*In words:* $k$ is the number of events observed, $\lambda$ the expected number; $\lambda^k / k!$ grows and then shrinks
with $k$, and $e^{-\lambda}$ makes the probabilities sum to 1. Poisson distributions are the classic model for **rare
events** over time or space.

🧮 **Math refresher — the exponential series.** $e^{\lambda} = \sum_{k=0}^{\infty} \frac{\lambda^k}{k!} = 1 + \lambda +
\frac{\lambda^2}{2} + \frac{\lambda^3}{6} + \cdots$ (the Taylor series of [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb)). Multiplying by $e^{-\lambda}$:
$\sum_k \frac{\lambda^k e^{-\lambda}}{k!} = e^{-\lambda}e^{\lambda} = 1$ — the PMF sums to 1. Its mean is $\lambda$
([Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb)).

In [ ]:
# ── 6.1 Poisson PMFs for λ = 1, 4, 10 ──
def poisson_pmf(k: int, lam: float) -> float:
    """p(k) = λ^k e^(−λ) / k!  (λ converted to float so that λ^k cannot overflow integer arithmetic)."""
    return float(lam) ** int(k) * np.exp(-lam) / factorial(int(k))


ks = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(6.4, 3.5))
for lam, colour in [(1, "tab:blue"), (4, "tab:red"), (10, "tab:green")]:
    pk = np.array([poisson_pmf(k, lam) for k in ks])
    ax.plot(ks, pk, "o--", ms=5, mfc="none", color=colour, label=f"λ = {lam}")
    print(f"λ = {lam:2d}: Σ p(k) over k ≤ 50 = {sum(poisson_pmf(k, lam) for k in range(51)):.10f};  "
          f"Σ k p(k) over k ≤ 80 = {sum(k * poisson_pmf(k, lam) for k in range(81)):.4f};  "
          f"max |ours − scipy| = {np.abs(pk - stats.poisson.pmf(ks, lam)).max():.1e}")
ax.set_xlabel("k (number of events)"); ax.set_ylabel("p(k)"); ax.set_title("Poisson(λ): larger λ shifts right and spreads out")
ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** For each $\lambda$ the probabilities add up to 1 (to ten decimals, summing to $k = 50$) and
the mean $\sum_k k\,p(k)$ equals $\lambda$; our formula agrees with SciPy to about $10^{-16}$. The picture: with
$\lambda = 1$ the most likely counts are 0 and 1 (both $e^{-1} = 0.368$) and large counts are very rare; as $\lambda$
increases the distribution shifts right and spreads out, centring around higher $k$.

### 6.1 A real-world scenario: deaths by horse kick

One of the first applications of the Poisson distribution was by the statistician **Ladislaus von Bortkiewicz**, in his
1898 book *The Law of Small Numbers*. He investigated accidental deaths by horse kick of soldiers in the Prussian army
in the late 1800s. The classic table summarises 20 years of data for 10 army corps — equivalent to 200 years of
observations of one corps. A mean of 0.61 soldiers per corps died from horse kicks each year; most years no soldier
died, but in one tragic year four soldiers of the same corps were killed. Let $X \sim \text{Poisson}(\lambda = 0.61)$
denote the number of deaths per year for a single corps; how well does the model fit?

In [ ]:
# ── 6.2 Bortkiewicz's horse-kick data (classic table, typed in) vs Poisson(0.61) ──
deaths = np.array([0, 1, 2, 3, 4])
corps_years = np.array([109, 65, 22, 3, 1])                    # number of corps-years with that many deaths
n_cy = corps_years.sum()
lam_hat = np.sum(deaths * corps_years) / n_cy                   # average deaths per corps-year
expected = n_cy * np.array([poisson_pmf(k, lam_hat) for k in deaths])
table_hk = pd.DataFrame({"deaths k": deaths, "observed": corps_years, "Poisson expected": expected.round(1),
                         "observed fraction": (corps_years / n_cy).round(3), "p(k)": (expected / n_cy).round(3)})
print(f"{n_cy} corps-years, {np.sum(deaths * corps_years)} deaths → mean λ = {lam_hat:.2f}")
print(table_hk.to_string(index=False))
print(f"P(X ≥ 5) = {1 - expected.sum() / n_cy:.5f} → expected {n_cy * (1 - expected.sum() / n_cy):.2f} corps-years with 5 or more deaths")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(deaths - 0.18, corps_years, width=0.36, color="tab:gray", label="observed (Bortkiewicz)")
ax.bar(deaths + 0.18, expected, width=0.36, color="tab:red", alpha=0.7, label="Poisson(0.61) × 200")
ax.set_xlabel("deaths per corps per year"); ax.set_ylabel("number of corps-years"); ax.legend()
ax.set_title("the law of small numbers: horse-kick deaths are Poisson")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 200 corps-years contain $65 + 2 \cdot 22 + 3 \cdot 3 + 4 \cdot 1 = 122$ deaths, a mean
of $\lambda = 0.61$. The Poisson model with that single parameter predicts 108.7, 66.3, 20.2, 4.1 and 0.6 corps-years
with 0–4 deaths; Bortkiewicz observed 109, 65, 22, 3 and 1 — a remarkably close fit ([Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb) shows how to test it
formally). The fit works because each soldier had a tiny, roughly constant risk and deaths happened independently —
the conditions behind the Poisson, as the next cell makes precise.

**The Poisson as a limit of the binomial.** Divide the interval into $n$ tiny pieces, each with an event with
probability $\alpha = \lambda / n$ (independently). The count is $\text{Binomial}(n, \lambda/n)$, and as $n \to \infty$:

$$\binom{n}{k}\Big(\frac{\lambda}{n}\Big)^k\Big(1 - \frac{\lambda}{n}\Big)^{n-k} = \underbrace{\frac{n(n-1)\cdots(n-k+1)}{n^k}}_{\to 1}\cdot\frac{\lambda^k}{k!}\cdot\underbrace{\Big(1 - \frac{\lambda}{n}\Big)^{n}}_{\to e^{-\lambda}}\cdot\underbrace{\Big(1 - \frac{\lambda}{n}\Big)^{-k}}_{\to 1} \;\longrightarrow\; \frac{\lambda^k e^{-\lambda}}{k!}.$$

*In words:* write $\binom nk = \frac{n(n-1)\cdots(n-k+1)}{k!}$ and regroup. The first fraction has $k$ factors each close
to 1 when $n$ is huge; $(1 - \lambda/n)^n \to e^{-\lambda}$ is the famous limit of [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb); and the last factor tends to 1.
**Many trials, each with a small success probability, give a Poisson count.**

In [ ]:
# ── 6.3 Binomial(n, λ/n) → Poisson(λ) ──
lam = 4.0
k_axis = np.arange(0, 16)
for n_trials in [10, 100, 1_000, 10_000]:
    diff = np.abs(stats.binom.pmf(k_axis, n_trials, lam / n_trials) - stats.poisson.pmf(k_axis, lam)).max()
    print(f"n = {n_trials:6,}: max_k |Binomial(n, 4/n) − Poisson(4)| = {diff:.5f}")
print("(1 − λ/n)^n for n = 10, 1000, 100000:", [round((1 - lam / m) ** m, 6) for m in (10, 1_000, 100_000)],
      f"  e^(−4) = {np.exp(-lam):.6f}")
fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.plot(k_axis, stats.poisson.pmf(k_axis, lam), "ko-", ms=4, label="Poisson(4)")
for n_trials, colour in [(10, "tab:orange"), (40, "tab:blue")]:
    ax.plot(k_axis, stats.binom.pmf(k_axis, n_trials, lam / n_trials), "s--", ms=3, color=colour, label=f"Binomial({n_trials}, 4/{n_trials})")
ax.set_xlabel("k"); ax.set_ylabel("p(k)"); ax.set_title("many rare trials → Poisson"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The largest difference between the two PMFs shrinks from 0.055 with $n = 10$ trials to
0.004, 0.0004 and 0.00004 as $n$ grows tenfold each time, and $(1 - 4/n)^n$ approaches $e^{-4} = 0.0183$.
Binomial(10, 0.4) is visibly narrower than Poisson(4); Binomial(40, 0.1) is already close. That is why the Poisson fits
any count of rare, independent events: deaths among many soldiers, mutations among billions of bases, reads from one
gene among millions of reads.

### 🌉 Bridge — RNA-seq read counts are (approximately) Poisson

In RNA-seq the number of reads mapping to each gene is a discrete random variable. Each sequenced fragment comes from
gene $G_i$ with a tiny probability, and there are millions of fragments — the binomial-to-Poisson limit — so the read
count of gene $G_i$ is modelled as

$$X_i \sim \text{Poisson}(\lambda_i), \qquad P(X_i = k) = \frac{e^{-\lambda_i}\lambda_i^k}{k!},$$

*In words:* $X_i$ is the observed count of gene $G_i$ and $\lambda_i$ its expected expression level. Marioni *et al.*
(2008) showed that counts from **technical replicates** (the same RNA sequenced twice) are well described by a
Poisson. The signature of a Poisson is that the **variance equals the mean**. **Biological replicates** (different
cells, mice or patients) vary more — their counts are **over-dispersed**, variance > mean — and are modelled with the
negative binomial (DESeq2, edgeR; [BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)); single-cell analysis pipelines build on the same count models ([BiologyToML 30 · Single-cell RNA-seq](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)). The toolkit's single-cell simulator shows both cases.

In [ ]:
# ── 6.4 Single-cell counts: Poisson (variance = mean) vs over-dispersed (variance > mean), SIMULATED ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, phi in zip(axes, [0.0, 0.5]):
    sim = simulate_cell_counts(overdispersion=phi)
    t_cells = sim["counts"][sim["labels"] == "T cell"]
    j = sim["genes"].index("IL7R")
    x = t_cells[:, j]
    m, v = x.mean(), x.var(ddof=1)
    kk = np.arange(0, x.max() + 1)
    ax.bar(kk, np.bincount(x, minlength=kk.size) / x.size, color="lightgray", edgecolor="gray", label="160 T cells")
    ax.plot(kk, stats.poisson.pmf(kk, m), "o-", color="tab:red", ms=3, label=f"Poisson(mean = {m:.1f})")
    ax.set_xlabel("IL7R molecules per cell"); ax.set_title(f"overdispersion φ = {phi}: mean {m:.1f}, variance {v:.1f}")
    ax.legend(fontsize=7)
    print(f"φ = {phi}: IL7R in T cells — mean {m:.2f}, variance {v:.2f}, variance/mean {v / m:.2f}")
    all_means = sim["counts"][sim["labels"] == "T cell"].mean(axis=0)
    all_vars = sim["counts"][sim["labels"] == "T cell"].var(axis=0, ddof=1)
    print(f"φ = {phi}: median variance/mean ratio over the 40 genes in T cells = {np.median(all_vars / all_means):.2f}")
axes[0].set_ylabel("fraction of cells")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: without over-dispersion the counts of the T-cell marker *IL7R* follow the Poisson PMF
with the same mean closely — mean 9.2, variance 9.7 — and the median variance/mean ratio over all 40 genes is 1.02.
Right: with over-dispersion φ = 0.5 (each cell's rate is itself random, as for real biological variation) the histogram
is far wider than the Poisson — more near-zero cells and a long tail up to 46 — and the variance (51.6) is about five
times the mean (10.0). The excess grows with the mean (the variance is $\lambda + \varphi\lambda^2$), so the median
ratio over all 40 genes, most of them weakly expressed, is a more modest 1.60.
Checking "variance ≈ mean?" is the first diagnostic for count data; [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) makes means and variances precise and
[Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) estimates $\lambda$ from data.

✏️ **Your turn.** (1) For the horse kicks, what is $P(X \ge 2)$ under Poisson(0.61)? How many of the 200 corps-years
does that predict? (2) A genomic region of 10 kb has a mutation rate of $10^{-8}$ per base per generation and is
followed for 1,000 generations in a population of 10,000 genomes. What is $\lambda$ for the total number of new
mutations in the region, and what is the probability of none?

---
## 7 · The geometric distribution: waiting for the first success

> 📍 **Where we are.** The binomial fixes the number of trials and counts successes. The geometric does the opposite:
> it counts the trials until the first success — a discrete waiting time.

The **geometric distribution** models the number of Bernoulli trials needed to get the first success. Parameter:
$\alpha \in (0, 1)$, the probability of success in each trial. Support: $k = 1, 2, 3, \ldots$ — the trial on which the
first success occurs:

$$p(k) = (1 - \alpha)^{k-1}\,\alpha \qquad \forall k \in \Omega = \{1, 2, 3, \ldots\}.$$

*In words:* the probability that the $k$-th coin flip is the first head (probability of heads $\alpha$) is the
probability of $k - 1$ tails in a row, $(1-\alpha)^{k-1}$, times the probability of a head on flip $k$ (independent
flips, product rule). The probabilities add up to 1 by the **geometric series**: $\sum_{k \ge 1}(1-\alpha)^{k-1}\alpha
= \alpha \cdot \frac{1}{1 - (1 - \alpha)} = 1$. Its mean is $1/\alpha$ ([Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb)).

**Memorylessness.** The probability of having to wait more than $k$ trials is the probability of $k$ failures in a row,
$P(K > k) = (1 - \alpha)^k$. Therefore

$$P(K > m + j \mid K > m) = \frac{P(K > m + j)}{P(K > m)} = \frac{(1-\alpha)^{m+j}}{(1-\alpha)^m} = (1-\alpha)^j = P(K > j).$$

*In words:* having waited $m$ trials without success does not change how much longer you will wait — the process has
no memory. (The numerator uses $\{K > m + j\} \subseteq \{K > m\}$, so their intersection is $\{K > m + j\}$.)

⚠️ Some books define the geometric variable as the number of **failures** before the first success (support
$0, 1, 2, \ldots$, PMF $(1-\alpha)^k\alpha$, mean $(1-\alpha)/\alpha = 1/\alpha - 1$). NumPy's `rng.geometric` and
SciPy's `stats.geom` use the *trials* version of this module. Example (Module 04): in random DNA a codon is a stop codon
with probability $\alpha = 3/64$, so in one reading frame the first stop is, on average, codon number $64/3 \approx 21.3$,
preceded by $61/3 \approx 20.3$ non-stop codons — which is why a long open reading frame is evidence of a real gene.

In [ ]:
# ── 7.1 Geometric PMFs, and SIMULATED waiting times for a mutation (illustrative α) ──
def geometric_pmf(k: int, alpha: float) -> float:
    """p(k) = (1 − α)^(k − 1) α for k = 1, 2, 3, …"""
    return (1 - alpha) ** (k - 1) * alpha


k_geo = np.arange(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for a, colour in [(0.2, "tab:blue"), (0.5, "tab:red"), (0.8, "tab:green")]:
    axes[0].plot(k_geo, [geometric_pmf(k, a) for k in k_geo], "o--", mfc="none", color=colour, label=f"α = {a}")
    print(f"α = {a}: Σ p(k) for k ≤ 200 = {sum(geometric_pmf(k, a) for k in range(1, 201)):.6f}, "
          f"p(1) = {geometric_pmf(1, a)}, matches scipy: {np.allclose([geometric_pmf(k, a) for k in k_geo], stats.geom.pmf(k_geo, a))}")
axes[0].set_xlabel("k (trial of the first success)"); axes[0].set_ylabel("p(k)"); axes[0].legend()
axes[0].set_title("Geometric(α): larger α → sharp drop; smaller α → slow decay")

rng = np.random.default_rng(SEED + 71)
alpha_mut = 0.05                                   # P(a lineage acquires the mutation in one generation), illustrative
waits = rng.geometric(alpha_mut, size=50_000)
kk = np.arange(1, 101)
axes[1].hist(waits, bins=np.arange(0.5, 101.5), density=True, color="lightgray", edgecolor="none", label="50,000 SIMULATED lineages")
axes[1].plot(kk, [geometric_pmf(k, alpha_mut) for k in kk], color="tab:red", label="Geometric(0.05) PMF")
axes[1].set_xlabel("generations until the first mutation"); axes[1].set_title("waiting time for a mutation"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"mean waiting time: simulated {waits.mean():.2f}, theory 1/α = {1 / alpha_mut:.0f}")
alpha_stop = 3 / 64                                          # P(a random codon is TAA, TAG or TGA)
print(f"random DNA: first stop codon at codon {1 / alpha_stop:.1f} on average (trials version), "
      f"after {(1 - alpha_stop) / alpha_stop:.1f} non-stop codons (failures version)")
still_waiting = waits[waits > 10] - 10                       # lineages with no mutation in the first 10 generations
print(f"memorylessness: P(K > 20) = {np.mean(waits > 20):.4f};  P(K > 30 | K > 10) = {np.mean(still_waiting > 20):.4f};"
      f"  theory (0.95)^20 = {0.95 ** 20:.4f}")

🔍 **What just happened?** Left: with $\alpha = 0.8$ success is likely early, so $p(1) = 0.8$ and the PMF drops sharply;
with $\alpha = 0.2$ more trials are expected and the PMF decays slowly. All three PMFs sum to 1 and match SciPy. Right:
50,000 simulated lineages, each acquiring a mutation with (illustrative) probability 0.05 per generation, wait on
average about 20 generations ($1/\alpha$), and the histogram follows the geometric PMF. The memorylessness check: the
fraction of all lineages that wait more than 20 generations and the fraction of lineages *already mutation-free after
10 generations* that wait more than 20 further generations are both about 0.36 ($0.95^{20} = 0.3585$). The stop-codon
line applies the same distribution to reading frames: 21.3 codons until the first stop, 20.3 non-stop codons before it. The geometric is the
discrete cousin of the exponential distribution of Section 10.

### 📘 Lecture notes — well-known PMFs

| Distribution | Support $\Omega$ | PMF $p(k)$ | Parameter(s) | Mean | Example scenario |
|---|---|---|---|---|---|
| Bernoulli | $\{0, 1\}$ | $\alpha^k(1-\alpha)^{1-k}$ | $0 \le \alpha \le 1$ | $\alpha$ | one (biased) coin flip; carrier or not |
| Multinoulli | $\{1, \ldots, k\}$ | $\prod_i p_i^{\delta(x,i)}$ | $p_1, \ldots, p_k$ | — | one base A/C/G/T |
| Binomial | $\{0, \ldots, n\}$ | $\binom nk \alpha^k(1-\alpha)^{n-k}$ | $n$, $\alpha$ | $n\alpha$ | heads in $n$ flips; correct reads among $n$ |
| Poisson | $\{0, 1, 2, \ldots\}$ | $\lambda^k e^{-\lambda}/k!$ | $\lambda > 0$ | $\lambda$ | rare events per interval; horse kicks; read counts |
| Geometric | $\{1, 2, \ldots\}$ | $(1-\alpha)^{k-1}\alpha$ | $0 < \alpha < 1$ | $1/\alpha$ | trial of the first head; generations until a mutation |

* **Bernoulli** mean and variance: $E[X] = 1 \cdot P(X{=}1) + 0 \cdot P(X{=}0) = \alpha$;
  $V(X) = E[X^2] - (E[X])^2 = 1^2 \cdot \alpha + 0^2 \cdot (1 - \alpha) - \alpha^2 = \alpha(1 - \alpha)$.
* **Binomial** plots: $n = 40$ with $\alpha = 0.1$ (outcomes near 0), $0.3$ (peak moves right), $0.5$ (symmetric
  around $n/2$); $N = 15$ with $\mu \in \{0.1, 0.4, 0.75\}$ ($\mu$ = success probability).
* **Poisson:** $\lambda$ = expected number of events; plots for $\lambda = 1, 4, 10$ (shift right and spread out as
  $\lambda$ grows). Bortkiewicz: 200 corps-years, mean 0.61 deaths, $X \sim \text{Poisson}(0.61)$; observed 109, 65, 22,
  3, 1 vs expected 108.7, 66.3, 20.2, 4.1, 0.6. Limit of $\text{Binomial}(n, \lambda/n)$.
* **Geometric:** larger $\alpha$ (0.8): sharp drop; smaller $\alpha$ (0.2): slower decay; memoryless.

> 🎓 **Exam tip.** Identify the family from the *story*: fixed number of independent yes/no trials → binomial; count of
> rare independent events in an interval → Poisson; number of trials until the first success → geometric.

✏️ **Your turn.** (1) What is the probability that the first head appears **after** the 5th flip of a fair coin?
Use $P(K > k) = (1-\alpha)^k$. (2) A PCR reaction fails with probability 0.15 independently each time. What is the
probability that you need at least 3 attempts to get a successful reaction?

---
## 8 · Continuous random variables and probability density functions

> 📍 **Where we are.** For continuous quantities — grades, weight, height, rainfall in inches, a concentration — every
> single value has probability 0 (Module 16), so a list of point probabilities is useless. Probabilities now live on
> intervals, and they are computed as **areas** under a curve (integrals, [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb)).

**Definition.** A random variable $X$ is **continuous** if there exists a function $f_X$ with $f_X(x) \ge 0$ for all $x$
and $\int_{-\infty}^{\infty} f_X(x)\,dx = 1$, such that for every $a \le b$

$$P(a < X < b) = \int_a^b f_X(x)\,dx.$$

*In words:* the probability that $X$ falls between $a$ and $b$ is the **area under the curve** $f_X$ between $a$ and
$b$. The function $f_X$ is called the **probability density function** (PDF). We still have the same properties as for
PMFs — everything adds (now: integrates) to 1, nothing is negative — but we typically consider a *range* rather than a
single value.

The CDF and the PDF determine each other:

$$F_X(x) = \int_{-\infty}^{x} f_X(t)\,dt \qquad\text{and}\qquad f_X(x) = F_X'(x) \text{ at all points } x \text{ at which } F_X \text{ is differentiable}.$$

*In words:* the CDF is the area under the density up to $x$, and the density is the slope of the CDF — the fundamental
theorem of calculus ([Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb)) in probability clothes.

**Key concepts.**

* The total area under the curve is 1.
* Probabilities are represented by **areas, not by point values**: $f(x)$ itself is **not a probability**. It is a
  probability *per unit* of $x$ (e.g. per mg/dL), and it **can be larger than 1**.
* That is why the probability of a single value is zero, $P(X = x) = 0$, and why $P(a \le X \le b) = P(a < X < b)$.

**PMF vs PDF.** On a *discrete* sample space a singleton event $\{\omega\}$ has $P(\{\omega\}) = p(\omega)$. On a
*continuous* sample space consider an interval event $A = [x, x + \Delta x]$ with small $\Delta x$:

$$P(A) = \int_x^{x + \Delta x} p(\omega)\,d\omega \approx p(x)\,\Delta x.$$

*In words:* over a short interval the density is almost constant, so the area is a thin rectangle of height $p(x)$ and
width $\Delta x$. The density times a small width is a probability; the density alone is not.

In [ ]:
# ── 8.1 Probability = area: a bell-shaped density, and P([x, x + Δx]) ≈ f(x)Δx ──
density = stats.norm(loc=1.5, scale=1.0)                     # any density works; Section 9 gives this one's formula
a, b = 1.0, 2.5
area, _ = quad(density.pdf, a, b)
print(f"P({a} ≤ X ≤ {b}) = ∫ f(x) dx = {area:.4f}   (CDF difference F(b) − F(a) = {density.cdf(b) - density.cdf(a):.4f})")
print(f"total area ∫ f = {quad(density.pdf, -np.inf, np.inf)[0]:.6f}")
x0 = 2.0
for dx in [1.0, 0.1, 0.01, 0.001]:
    exact, _ = quad(density.pdf, x0, x0 + dx)
    print(f"Δx = {dx:<5}: P([2, 2 + Δx]) = {exact:.6f}   f(2)·Δx = {density.pdf(x0) * dx:.6f}   ratio {exact / (density.pdf(x0) * dx):.4f}")

xs = np.linspace(-2, 5, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.plot(xs, density.pdf(xs), color="mediumvioletred", lw=2)
mask = (xs >= a) & (xs <= b)
ax.fill_between(xs[mask], density.pdf(xs[mask]), color="gold", alpha=0.9, label=f"P({a} ≤ X ≤ {b}) = area = {area:.3f}")
ax.set_xlabel("x"); ax.set_ylabel("f(x)  (density)"); ax.set_title("probability over a range is an area"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The area under this bell-shaped density between 1.0 and 2.5 is 0.5328, computed both by
numerical integration (`quad`) and as the CDF difference $F(2.5) - F(1.0)$; the total area is 1. The table shows
$P([x, x + \Delta x]) \approx f(x)\Delta x$ getting better as $\Delta x$ shrinks: the ratio goes from 0.69 for $\Delta x = 1$
to 0.974, 0.9975 and 0.9997 for $\Delta x = 0.1$, 0.01, 0.001 — and both sides go to 0, so $P(X = 2) = 0$ exactly.

### 8.1 The uniform distribution

$X$ has a **Uniform**$(a, b)$ distribution, written $X \sim \text{Uniform}(a, b)$ with $a < b$, if all outcomes with
values between $a$ and $b$ are equally probable:

$$f(x) = \begin{cases} \dfrac{1}{b - a} & x \in [a, b]\\[4pt] 0 & \text{otherwise}, \end{cases} \qquad F(x) = \begin{cases} 0 & x < a\\ \dfrac{x - a}{b - a} & x \in [a, b]\\ 1 & x > b. \end{cases}$$

*In words:* the density is a flat plateau of height $1/(b - a)$ — chosen so the rectangle's area is
$(b - a) \cdot \frac{1}{b-a} = 1$ — and the CDF rises linearly from 0 at $a$ to 1 at $b$ (the area of the part of the
rectangle to the left of $x$). The special case $\text{Uniform}(0, 1)$ has $f_X(x) = 1$ for $0 \le x \le 1$ (0
otherwise) and $F_X(x) = x$ on $[0, 1]$: it captures the idea of choosing a point at random between 0 and 1 (Module 16,
Section 8). Clearly $f_X(x) \ge 0$ and $\int f_X(x)\,dx = 1$.

In [ ]:
# ── 8.2 Uniform PDFs and CDFs; numerical checks ──
def uniform_pdf(x, a: float, b: float):
    """1/(b − a) on [a, b], 0 elsewhere (works on arrays)."""
    x = np.asarray(x, dtype=float)
    return np.where((x >= a) & (x <= b), 1.0 / (b - a), 0.0)


def uniform_cdf(x, a: float, b: float):
    """0 below a, (x − a)/(b − a) on [a, b], 1 above b."""
    return np.clip((np.asarray(x, dtype=float) - a) / (b - a), 0.0, 1.0)


xs = np.linspace(-6, 6, 1201)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
for (a_, b_), colour in [((-5, 5), "tab:blue"), ((-1, 3), "tab:red"), ((-3, 3), "tab:green")]:
    axes[0].plot(xs, uniform_pdf(xs, a_, b_), color=colour, label=f"a = {a_}, b = {b_}")
    axes[1].plot(xs, uniform_cdf(xs, a_, b_), color=colour, label=f"a = {a_}, b = {b_}")
    total = quad(lambda t: uniform_pdf(t, a_, b_), a_ - 1, b_ + 1, points=[a_, b_])[0]
    print(f"Uniform({a_}, {b_}): height 1/(b − a) = {1 / (b_ - a_):.4f}, area = {total:.4f}, "
          f"F(0) = {uniform_cdf(0, a_, b_):.3f} (scipy {stats.uniform(a_, b_ - a_).cdf(0):.3f})")
axes[0].set_ylim(0, 0.5); axes[0].set_title("Uniform(a, b) densities"); axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x)")
axes[1].set_title("their CDFs"); axes[1].set_xlabel("x"); axes[1].set_ylabel("F(x)")
axes[0].legend(fontsize=7)
xs01 = np.linspace(-0.5, 1.5, 401)
axes[2].plot(xs01, uniform_pdf(xs01, 0, 1), color="0.4", ls="--", label="PDF f(x)")
axes[2].plot(xs01, uniform_cdf(xs01, 0, 1), color="k", lw=2.5, label="CDF F(x) = x")
axes[2].set_title("Uniform(0, 1)"); axes[2].set_xlabel("x"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three densities are rectangles of heights $1/10 = 0.1$, $1/4 = 0.25$ and $1/6 \approx
0.167$: the wider the interval, the lower the plateau, because every rectangle must have area 1 (the integrals print
1.0000). Their CDFs are ramps from 0 to 1 across $[a, b]$; for example $F(0) = 0.5$ for $\text{Uniform}(-5, 5)$ and
$0.25$ for $\text{Uniform}(-1, 3)$, matching SciPy. The right panel shows $\text{Uniform}(0, 1)$: density 1 on the unit
interval, CDF $F(x) = x$ — flat at 0, a straight ramp, then flat at 1.

**Discrete vs continuous probabilities.** Compare a **discrete** uniform distribution on three points (each with
probability 1/3, drawn as stems) with a **continuous** uniform distribution on an interval of length 0.7 (a plateau of
height $1/0.7 \approx 1.43$).

In [ ]:
# ── 8.3 Discrete vs continuous uniform: probabilities at points vs density over an interval ──
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2), sharey=True)
z_points = [-1.1, 0.3, 1.5]
axes[0].vlines(z_points, 0, 1 / 3, color="tab:blue", ls="--"); axes[0].plot(z_points, [1 / 3] * 3, "o", color="tab:red")
axes[0].set_xlim(-1.5, 2); axes[0].set_ylim(0, 2); axes[0].set_xlabel("z"); axes[0].set_ylabel("P(Z = z)")
axes[0].set_title("(a) discrete: three values, 1/3 each")
lo, hi = 0.9, 1.6
axes[1].plot([lo, hi], [1 / (hi - lo)] * 2, color="tab:red", lw=2)
axes[1].vlines([lo, hi], 0, 1 / (hi - lo), color="tab:blue", ls="--")
axes[1].set_xlim(-1.5, 2); axes[1].set_xlabel("x"); axes[1].set_ylabel("p(x)")
axes[1].set_title(f"(b) continuous: density {1 / (hi - lo):.2f} on [{lo}, {hi}]")
plt.tight_layout(); plt.show()
print(f"(a) Σ P(Z = z) = {3 * (1 / 3):.1f};  (b) area = height × width = {1 / (hi - lo):.4f} × {hi - lo:.1f} = {1 / (hi - lo) * (hi - lo):.1f}")

🔍 **What just happened?** Panel (a): all probability sits on three points, each with probability 1/3 — the heights
*are* probabilities and add to 1. Panel (b): the probability is spread evenly over $[0.9, 1.6]$; the plateau's height is
$1/0.7 = 1.43$, **larger than 1**, yet it is a perfectly valid density because the *area* is $1.43 \times 0.7 = 1$.
⚠️ A density above 1 is a common source of confusion — it simply means that the variable is concentrated on an interval
shorter than one unit.

### 📘 Lecture notes — probability density functions and the uniform distribution

**PDF:** $X$ is continuous if there is $f_X \ge 0$ with $\int_{-\infty}^{\infty} f_X(x)\,dx = 1$ and, for every
$a \le b$, $P(a < X < b) = \int_a^b f_X(x)\,dx$. Then $F_X(x) = \int_{-\infty}^x f_X(t)\,dt$ and $f_X(x) = F_X'(x)$
wherever $F_X$ is differentiable. Total area = 1; probabilities are areas, not point values; $f(x)$ is not a probability
(it can exceed 1); $P(X = x)$ is (usually) zero. Examples of continuous variables: grades, weight, height, rainfall.

**PMF vs PDF:** discrete singleton $P(\{\omega\}) = p(\omega)$; continuous small interval
$P([x, x + \Delta x]) = \int_x^{x+\Delta x} p(\omega)\,d\omega \approx p(x)\Delta x$.

**Uniform**$(a, b)$, $a < b$: $f(x) = \frac{1}{b-a}$ on $[a, b]$, 0 otherwise; $F(x) = 0$ ($x < a$), $\frac{x-a}{b-a}$
($a \le x \le b$), 1 ($x > b$). Plots for $(a, b) = (-5, 5), (-1, 3), (-3, 3)$. **Uniform(0, 1):** $f = 1$ on $[0, 1]$,
$F(x) = x$ — "choose a point at random between 0 and 1". Discrete uniform (equal point masses) vs continuous uniform
(flat density, possibly above 1).

> 🎓 **Exam tip.** To get a probability from a density, *integrate*; to get a density from a CDF, *differentiate*. Never
> report $f(x)$ as "the probability of $x$".

✏️ **Your turn.** (1) For $X \sim \text{Uniform}(-1, 3)$ compute $P(0 \le X \le 2.5)$ with `uniform_cdf` and by `quad`.
(2) A cell divides at a time uniformly distributed between 18 and 26 hours after its birth. What are the density's
height and the probability that it divides within 20 hours?

---
## 9 · The Gaussian (normal) distribution

> 📍 **Where we are.** The bell curve of Section 8 has a name and a formula. It is the single most important
> distribution in statistics: many phenomena in nature are approximately normal, and the central limit theorem
> ([Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb)) explains why.

$X$ has a **Normal** (or **Gaussian**) distribution with parameters $\mu$ and $\sigma$, denoted $X \sim N(\mu, \sigma^2)$,
if

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\Big\{-\frac{1}{2\sigma^2}(x - \mu)^2\Big\}, \qquad x \in \mathbb{R}.$$

*In words:* the density is highest at $x = \mu$ and falls off symmetrically on both sides; the exponent measures the
squared distance from $\mu$ in units of $\sigma^2$, so values more than a few $\sigma$ away are very unlikely; the factor
$1/(\sigma\sqrt{2\pi})$ makes the total area exactly 1. It is parameterised by the **mean** (centre) $\mu \in \mathbb{R}$
and the **standard deviation** (spread) $\sigma > 0$. We say $X$ has a **standard Normal** distribution if $\mu = 0$ and
$\sigma = 1$; its density is often written $\varphi(z)$.

🧮 **Math refresher — exp, π and areas.** $\exp\{u\} = e^u$ with $e \approx 2.718$ ([Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb)); $e^0 = 1$ and $e^u \to 0$
as $u \to -\infty$. $\pi \approx 3.1416$. That the area is exactly 1 follows from the famous integral
$\int_{-\infty}^{\infty} e^{-z^2/2}\,dz = \sqrt{2\pi}$; we check it numerically.

Example scenarios: most natural phenomena — average daily temperature, the height of adults, measurement error, and
**quantitative traits** influenced by many genes with small effects (a sum of many small random contributions is
approximately normal — the central limit theorem).

In [ ]:
# ── 9.1 Gaussian densities: the formula, the standard normal, and three (μ, σ) pairs ──
def gaussian_pdf(x, mu: float, sigma: float):
    """N(μ, σ²) density: exp(−(x − μ)²/(2σ²)) / (σ √(2π))."""
    x = np.asarray(x, dtype=float)
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))


xs = np.linspace(-6, 6, 1201)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(xs, gaussian_pdf(xs, 0, 1), color="k")
axes[0].set_xlim(-3, 3); axes[0].set_title("density of a standard Normal, φ(z)"); axes[0].set_xlabel("z")
for (mu, sigma), colour in [((0, 0.5), "tab:blue"), ((0, 1), "tab:red"), ((-2, 0.75), "tab:green")]:
    axes[1].plot(xs, gaussian_pdf(xs, mu, sigma), color=colour, label=f"μ = {mu}, σ = {sigma}")
    area = quad(lambda t: gaussian_pdf(t, mu, sigma), -np.inf, np.inf)[0]
    print(f"N({mu}, {sigma}²): peak 1/(σ√2π) = {gaussian_pdf(mu, mu, sigma):.4f}, area = {area:.6f}, "
          f"max |ours − scipy| = {np.abs(gaussian_pdf(xs, mu, sigma) - stats.norm.pdf(xs, mu, sigma)).max():.1e}")
axes[1].set_xlabel("x"); axes[1].set_ylabel("f(x)"); axes[1].set_title("μ moves the centre, σ the spread"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"∫ exp(−z²/2) dz = {quad(lambda z: np.exp(-z ** 2 / 2), -np.inf, np.inf)[0]:.6f}  vs √(2π) = {np.sqrt(2 * np.pi):.6f}")

🔍 **What just happened?** The standard normal (left) is the familiar bell centred at 0. On the right, $\mu$ shifts the
bell ($\mu = -2$, green) and $\sigma$ changes its width *and* its height: the narrow $\sigma = 0.5$ curve peaks at
0.7979, the $\sigma = 1$ curve at 0.3989, the $\sigma = 0.75$ curve at 0.5319 — narrower bells are taller, because the
area is always 1 (all three integrate to 1.000000). Our formula matches SciPy to about $10^{-16}$, and
$\int e^{-z^2/2}dz = 2.506628 = \sqrt{2\pi}$.

**The Gaussian CDF has no closed form.** For continuous variables the CDF is still $p(X \le x) = \int_{-\infty}^x f(z)\,dz$,
and for the Gaussian

$$F(x) = \int_{-\infty}^{x}\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(z-\mu)^2}{2\sigma^2}}\,dz = \Phi\Big(\frac{x - \mu}{\sigma}\Big), \qquad \Phi(z) = \frac12\Big[1 + \operatorname{erf}\Big(\frac{z}{\sqrt2}\Big)\Big].$$

*In words:* the integral cannot be written with elementary functions, so mathematicians gave the standard normal CDF a
name, $\Phi$, and expressed it through the **error function** erf, which computers evaluate numerically. Any Gaussian
reduces to the standard one through the **z-score** $z = (x - \mu)/\sigma$ — "how many standard deviations above the
mean". In practice: `scipy.stats.norm.cdf`.

In [ ]:
# ── 9.2 Φ three ways, the 68–95–99.7 rule, Gaussian CDFs, and a popular approximation ──
for z in [-1.0, 0.0, 1.0, 1.96, 2.5]:
    by_integral = quad(lambda t: gaussian_pdf(t, 0, 1), -np.inf, z)[0]
    by_erf = 0.5 * (1 + erf(z / np.sqrt(2)))
    print(f"Φ({z:5.2f}): integral {by_integral:.6f} | erf {by_erf:.6f} | scipy {stats.norm.cdf(z):.6f}")
for k in (1, 2, 3):
    print(f"P(μ − {k}σ ≤ X ≤ μ + {k}σ) = Φ({k}) − Φ(−{k}) = {stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")


def logistic_normal_cdf(z):
    """A closed-form approximation of Φ sometimes used for plotting: 1 / (1 + exp(−0.07056 z³ − 1.5976 z))."""
    return 1.0 / (1.0 + np.exp(-0.07056 * z ** 3 - 1.5976 * z))


zz = np.linspace(-6, 6, 2001)
print(f"largest error of the logistic approximation on [−6, 6]: {np.abs(logistic_normal_cdf(zz) - stats.norm.cdf(zz)).max():.1e}")
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for (mu, sigma), colour in [((0, 0.5), "tab:blue"), ((0, 1), "tab:red"), ((-2, 0.75), "tab:green")]:
    ax.plot(xs, stats.norm.cdf(xs, mu, sigma), color=colour, label=f"μ = {mu}, σ = {sigma}")
ax.set_xlabel("x"); ax.set_ylabel("F(x) = P(X ≤ x)"); ax.set_title("Gaussian CDFs: S-shaped, F(μ) = 1/2"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Numerical integration of the density, the erf formula and SciPy agree to six decimals:
$\Phi(-1) = 0.158655$, $\Phi(0) = 0.5$, $\Phi(1) = 0.841345$, $\Phi(1.96) = 0.975002$, $\Phi(2.5) = 0.993790$. The
**68–95–99.7 rule** follows: 68.27 % of a Gaussian lies within one $\sigma$ of the mean, 95.45 % within $2\sigma$,
99.73 % within $3\sigma$. Every Gaussian CDF is an S-shaped curve crossing 1/2 at $x = \mu$, steeper for smaller $\sigma$.
A closed-form logistic formula is sometimes used to *draw* such curves; it is accurate to about $10^{-4}$ — good for a
picture, not for a p-value.

**Two random variables.** The Gaussian extends to several variables. A 2-D Gaussian for $(x_1, x_2)$ has a mean for each
variable and a 2 × 2 **covariance matrix** describing their spreads and how they vary together ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)); its density is a
smooth hill over the plane whose contour lines are ellipses.

In [ ]:
# ── 9.3 A 2-D Gaussian density (preview of several random variables) ──
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)

mean2 = np.array([0.0, 1.0])
cov2 = np.array([[0.25, 0.3], [0.3, 4.0]])                     # σ1 = 0.5, σ2 = 2, correlation 0.3
g1, g2 = np.meshgrid(np.linspace(-1.5, 1.5, 80), np.linspace(-5, 7.5, 80))
dens = stats.multivariate_normal(mean2, cov2).pdf(np.dstack([g1, g2]))
fig = plt.figure(figsize=(10, 3.8))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax3.plot_wireframe(g1, g2, dens, rstride=4, cstride=4, color="0.4", lw=0.6)
ax3.contour(g1, g2, dens, zdir="z", offset=0, cmap="viridis", levels=6)
ax3.set_xlabel("x₁"); ax3.set_ylabel("x₂"); ax3.set_zlabel("p(x₁, x₂)"); ax3.set_title("2-D Gaussian density")
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(g1, g2, dens, levels=6, cmap="viridis")
ax2.plot(*mean2, "k+", ms=10); ax2.set_xlabel("x₁"); ax2.set_ylabel("x₂"); ax2.set_title("contours are ellipses")
plt.tight_layout(); plt.show()
cell_area = (g1[0, 1] - g1[0, 0]) * (g2[1, 0] - g2[0, 0])
print(f"volume under the surface on this grid ≈ {dens.sum() * cell_area:.3f} (the 2-D analogue of 'area = 1')")

🔍 **What just happened?** The wireframe is the joint density of two Gaussian variables with means 0 and 1, standard
deviations 0.5 and 2 and a modest positive correlation (0.3); the contour lines on the floor and on the right are
ellipses tilted slightly because the variables tend to move together. The **volume** under the surface is 1 (0.996 on
this finite grid) — probabilities of regions in the plane are volumes. [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) develops joint, marginal and conditional
distributions of several random variables.

### 🌉 Bridge — CDF thresholds for gene expression

Suppose the log₂ expression of a tumour-marker gene in **normal** tissue is approximately Gaussian (illustrative:
$\mu = 5$, $\sigma = 1$). A lab calls a sample "high" when the expression exceeds a threshold $t$. The probability that a
normal sample is (falsely) flagged is a CDF computation:

$$P(X > t) = 1 - F_X(t) = 1 - \Phi\Big(\frac{t - \mu}{\sigma}\Big).$$

*In words:* the probability of exceeding a threshold is one minus the probability of staying at or below it — the
complement rule of Module 16 applied to the event $\{X \le t\}$. Conversely, to flag only 1 % of normal samples we choose
$t$ as the **99th percentile** of the normal distribution, $t = F^{-1}(0.99)$ (the inverse of the CDF, `norm.ppf`).
This is how genes are called "highly expressed" relative to a reference distribution, and it is the seed of the
p-values of [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) and of the fold-change thresholds used by tools such as DESeq2 (Love *et al.* 2014).

In [ ]:
# ── 9.4 Tail probabilities and thresholds for a marker gene (ILLUSTRATIVE parameters, SIMULATED samples) ──
mu_n, sd_n = 5.0, 1.0                                       # log2 expression in normal tissue
for t in [6.0, 7.0, 7.5]:
    z = (t - mu_n) / sd_n
    print(f"threshold t = {t}: z = {z:.1f},  P(X > t) = 1 − Φ({z:.1f}) = {1 - stats.norm.cdf(z):.4f}")
t99 = stats.norm.ppf(0.99, mu_n, sd_n)
print(f"threshold that flags 1 % of normal samples: t = F⁻¹(0.99) = {t99:.3f}  (z = {stats.norm.ppf(0.99):.3f})")
rng = np.random.default_rng(SEED + 94)
normal_samples = rng.normal(mu_n, sd_n, 10_000)
print(f"10,000 SIMULATED normal samples: fraction above 7.5 = {np.mean(normal_samples > 7.5):.4f};  above t99 = {np.mean(normal_samples > t99):.4f}")

🔍 **What just happened?** A threshold one standard deviation above the normal mean (t = 6) flags 15.9 % of normal
samples; t = 7 (z = 2) flags 2.3 %; t = 7.5 (z = 2.5) flags 0.62 %. To flag exactly 1 % we need $t = 5 + 2.326 = 7.326$,
the 99th percentile. Ten thousand simulated normal samples give 0.57 % above 7.5 and 0.87 % above $t_{99}$ — within about 0.15
percentage points of the theory.
The same calculation, run for thousands of genes at once, raises the multiple-testing problem of [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb).

✏️ **Your turn.** (1) Using `stats.norm.cdf`, compute the probability that an adult height from $N(170, 7^2)$ cm lies
between 160 and 180 cm. (2) If tumour samples have marker expression $N(8, 1^2)$, what fraction of them exceeds the
threshold $t_{99} = 7.326$ (the **sensitivity** of the rule, Module 17)?

---
## 10 · The exponential distribution, a summary, and sampling

> 📍 **Where we are.** The Poisson counts rare events in an interval. The exponential describes the *waiting time*
> between them — the continuous counterpart of the geometric. Then we put all the distributions side by side.

The **exponential distribution** is parameterised by $\lambda > 0$, the average **rate** of arrivals over time:

$$f(x) = \lambda e^{-\lambda x} \quad (x \ge 0), \qquad f(x) = 0 \quad (x < 0), \qquad F(x) = 1 - e^{-\lambda x} \quad (x \ge 0).$$

*In words:* short waits are most likely and long waits become exponentially rarer; the CDF follows by integrating the
density from 0 to $x$: $\int_0^x \lambda e^{-\lambda t}dt = \big[-e^{-\lambda t}\big]_0^x = 1 - e^{-\lambda x}$. The mean
waiting time is $1/\lambda$. The exponential models the lifetimes of electronic components and the **waiting times
between rare events** — which can also be thought of as the **times between Poisson events**: if events occur at rate
$\lambda$ per unit time, the count in one unit is $\text{Poisson}(\lambda)$ and the gaps between events are
$\text{Exponential}(\lambda)$. Like the geometric, the exponential is **memoryless**:
$P(X > s + t \mid X > s) = e^{-\lambda(s+t)}/e^{-\lambda s} = e^{-\lambda t} = P(X > t)$.

Back to Bortkiewicz: if horse-kick deaths in a corps occur at rate $\lambda = 0.61$ per year, the number of deaths per
year is Poisson(0.61) and the time between two deaths is Exponential(0.61), with mean $1/0.61 \approx 1.64$ years. The
same picture describes **mutations along a genome** (a Poisson process along the sequence: counts per window are
Poisson, distances between neighbouring mutations exponential).

In [ ]:
# ── 10.1 Exponential densities and CDFs ──
def exponential_pdf(x, lam: float):
    """λ e^(−λx) for x ≥ 0, 0 for x < 0 (works on arrays)."""
    x = np.asarray(x, dtype=float)
    return np.where(x >= 0, lam * np.exp(-lam * np.clip(x, 0, None)), 0.0)


xs = np.linspace(-0.5, 5, 551)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
for lam, colour in [(0.5, "tab:blue"), (1.0, "tab:red"), (1.5, "tab:green")]:
    axes[0].plot(xs, exponential_pdf(xs, lam), color=colour, label=f"λ = {lam}")
    axes[1].plot(xs, np.where(xs >= 0, 1 - np.exp(-lam * np.clip(xs, 0, None)), 0), color=colour, label=f"λ = {lam}")
    area = quad(lambda t: exponential_pdf(t, lam), 0, np.inf)[0]
    mean = quad(lambda t: t * exponential_pdf(t, lam), 0, np.inf)[0]
    print(f"λ = {lam}: area = {area:.6f}, mean ∫ x f(x) dx = {mean:.4f} = 1/λ = {1 / lam:.4f}, f(0) = {lam}")
axes[0].set_xlabel("waiting time x"); axes[0].set_ylabel("f(x)"); axes[0].set_title("Exponential(λ) densities"); axes[0].legend()
axes[1].set_xlabel("waiting time x"); axes[1].set_ylabel("F(x)"); axes[1].set_title("CDFs 1 − e^(−λx)"); axes[1].legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each density starts at height $f(0) = \lambda$ (so $\lambda = 1.5$ gives a density above 1)
and decays exponentially; each integrates to 1 and has mean $1/\lambda$ = 2, 1 and 0.667. Higher rates mean shorter
typical waits and CDFs that rise faster.

In [ ]:
# ── 10.2 A SIMULATED Poisson process of horse-kick deaths: Poisson counts and exponential gaps ──
rng = np.random.default_rng(SEED + 102)
rate, years = 0.61, 10_000                                   # one corps followed for 10,000 years
gaps = rng.exponential(1 / rate, size=int(rate * years * 1.2))
times = np.cumsum(gaps)
times = times[times < years]                                 # death times in [0, 10,000) years
per_year = np.bincount(times.astype(int), minlength=years)   # deaths in each calendar year
kk = np.arange(0, 6)
print("deaths per year:   ", np.round(np.bincount(per_year, minlength=6)[:6] / years, 4))
print("Poisson(0.61) PMF: ", np.round(stats.poisson.pmf(kk, rate), 4))
print(f"mean gap between deaths {np.diff(times).mean():.3f} years (1/λ = {1 / rate:.3f});  "
      f"fraction of gaps > 3 years {np.mean(np.diff(times) > 3):.4f} (e^(−0.61·3) = {np.exp(-rate * 3):.4f})")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].bar(kk - 0.18, np.bincount(per_year, minlength=6)[:6] / years, width=0.36, color="tab:gray", label="simulated years")
axes[0].bar(kk + 0.18, stats.poisson.pmf(kk, rate), width=0.36, color="tab:red", alpha=0.7, label="Poisson(0.61)")
axes[0].set_xlabel("deaths in a year"); axes[0].set_ylabel("fraction"); axes[0].legend(); axes[0].set_title("counts per year")
axes[1].hist(np.diff(times), bins=60, range=(0, 9), density=True, color="lightgray", edgecolor="gray", label="simulated gaps")
axes[1].plot(np.linspace(0, 9, 200), exponential_pdf(np.linspace(0, 9, 200), rate), color="tab:red", label="Exponential(0.61)")
axes[1].set_xlabel("years between deaths"); axes[1].set_ylabel("density"); axes[1].legend(); axes[1].set_title("waiting times")
plt.tight_layout(); plt.show()

🔍 **What just happened?** We generated death *times* by adding up exponential gaps with mean $1/0.61 = 1.64$ years and
then simply counted the deaths in each calendar year. The yearly counts reproduce the Poisson(0.61) PMF
(simulated 0.547, 0.329, 0.100, 0.021 vs 0.543, 0.331, 0.101, 0.021), and the gaps reproduce the exponential density; the fraction of gaps longer than
3 years matches $e^{-0.61 \cdot 3} = 0.160$. **One process, two descriptions:** counts in fixed windows are Poisson,
waiting times between events are exponential.

### PMF vs PDF vs CDF

| | **PMF** | **PDF** | **CDF** |
|---|---|---|---|
| definition | $p(x) = P(X = x)$ | $P(a \le X \le b) = \int_a^b f(x)\,dx$ | $F(x) = P(X \le x)$ |
| meaning | direct probability | density (area under the curve gives probability) | accumulated probability up to $x$ |
| normalisation | $\sum_x p(x) = 1$ | $\int_{-\infty}^{\infty} f(x)\,dx = 1$ | $F(x) \in [0, 1]$, from 0 to 1 |
| for | discrete variables | continuous variables | both |
| relation | $F(x) = \sum_{x_i \le x} p(x_i)$ | $F(x) = \int_{-\infty}^x f(t)\,dt$, $f = F'$ | jumps = PMF; slope = PDF |

Finally, **sampling**: every distribution of this module can be simulated with `rng`, and with enough draws the
histogram of the samples approaches the PMF or PDF — the bridge from probability to data that Part F builds on.

In [ ]:
# ── 10.3 Sampling from every distribution of the module (SIMULATED, 20,000 draws each) ──
rng = np.random.default_rng(SEED + 103)
n_draws = 20_000
discrete = [("Bernoulli(0.3)", rng.binomial(1, 0.3, n_draws), lambda k: stats.bernoulli.pmf(k, 0.3)),
            ("Binomial(40, 0.3)", rng.binomial(40, 0.3, n_draws), lambda k: stats.binom.pmf(k, 40, 0.3)),
            ("Poisson(4)", rng.poisson(4, n_draws), lambda k: stats.poisson.pmf(k, 4)),
            ("Geometric(0.2)", rng.geometric(0.2, n_draws), lambda k: stats.geom.pmf(k, 0.2))]
continuous = [("Uniform(−1, 3)", rng.uniform(-1, 3, n_draws), lambda x: uniform_pdf(x, -1, 3)),
              ("N(0, 1)", rng.normal(0, 1, n_draws), lambda x: gaussian_pdf(x, 0, 1)),
              ("N(−2, 0.75²)", rng.normal(-2, 0.75, n_draws), lambda x: gaussian_pdf(x, -2, 0.75)),
              ("Exponential(1.5)", rng.exponential(1 / 1.5, n_draws), lambda x: exponential_pdf(x, 1.5))]
fig, axes = plt.subplots(2, 4, figsize=(12, 5.4))
for ax, (name, sample, pmf) in zip(axes[0], discrete):
    k = np.arange(sample.min(), min(sample.max(), 30) + 1)
    ax.bar(k, np.array([np.mean(sample == v) for v in k]), color="lightgray", edgecolor="gray")
    ax.plot(k, pmf(k), "o", color="tab:red", ms=3)
    ax.set_title(f"{name}\nsample mean {sample.mean():.3f}", fontsize=9)
for ax, (name, sample, pdf) in zip(axes[1], continuous):
    ax.hist(sample, bins=50, density=True, color="lightgray", edgecolor="gray")
    grid_x = np.linspace(sample.min(), sample.max(), 300)
    ax.plot(grid_x, pdf(grid_x), color="tab:red")
    ax.set_title(f"{name}\nsample mean {sample.mean():.3f}", fontsize=9)
axes[0, 0].set_ylabel("fraction (bars) / PMF (dots)"); axes[1, 0].set_ylabel("density")
fig.suptitle("20,000 random draws per distribution: histograms approach the PMF / PDF (red)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** `rng.binomial`, `rng.poisson`, `rng.geometric`, `rng.uniform`, `rng.normal` and
`rng.exponential` each drew 20,000 values. In every panel the grey histogram sits on the red PMF dots or PDF curve, and
the sample means (in the titles) land close to the theoretical means $0.3$, $40 \times 0.3 = 12$, $4$, $1/0.2 = 5$,
$1$, $0$, $-2$ and $1/1.5 = 0.667$. Note the `rng.exponential` convention: NumPy asks for the **scale** $1/\lambda$, not
the rate $\lambda$ — a classic source of bugs.

### 📘 Lecture notes — the Gaussian, the exponential and the PMF/PDF/CDF summary

**Gaussian** $X \sim N(\mu, \sigma^2)$: $f(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\{-\frac{1}{2\sigma^2}(x-\mu)^2\}$,
$x \in \mathbb{R}$; $\mu \in \mathbb{R}$ the centre (mean), $\sigma > 0$ the spread (standard deviation). Standard Normal:
$\mu = 0$, $\sigma = 1$. Plots for $(\mu, \sigma) = (0, 0.5), (0, 1), (-2, 0.75)$. Many natural phenomena (daily
temperature, adult height) are approximately Normal; the Central Limit Theorem says a sum of random variables is
approximately Normal. CDF: $p(X \le x) = \int_{-\infty}^x f(z)\,dz$ — no closed form (use $\Phi$, erf,
`scipy.stats.norm.cdf`); 68–95–99.7 rule; z-scores. A 2-D Gaussian describes two random variables $x_1, x_2$ jointly.

**Exponential**$(\lambda)$, $\lambda > 0$ the average rate of arrivals: $f(x) = \lambda e^{-\lambda x}$ for $x \ge 0$
(0 otherwise); $F(x) = 1 - e^{-\lambda x}$; mean $1/\lambda$; lifetimes and **waiting times between rare events** = times
between Poisson events (horse kicks: Poisson(0.61) counts per year, Exponential(0.61) gaps, mean 1.64 years). Plots for
$\lambda = 0.5, 1, 1.5$.

**PMF vs PDF vs CDF:** PMF $p(x) = P(X = x)$, direct probability, $\sum_x p(x) = 1$, discrete; PDF
$P(a \le X \le b) = \int_a^b f(x)dx$, density, $\int f = 1$, continuous; CDF $F(x) = P(X \le x)$, accumulated
probability, $F(x) \in [0, 1]$, both. **Threshold decisions:** $P(X > x) = 1 - F_X(x)$.

> 🎓 **Exam tip.** For "the probability that $X$ exceeds $t$" always reach for $1 - F(t)$, and for continuous variables
> do not worry about $>$ vs $\ge$. For the exponential remember both the rate form $\lambda e^{-\lambda x}$ and that
> NumPy's `rng.exponential` takes the scale $1/\lambda$.

✏️ **Your turn.** (1) Mutations fall along a chromosome at an average rate of 1 per 5 kb (illustrative). What is the
probability that two neighbouring mutations are more than 10 kb apart? Check by simulating 100,000 gaps. (2) In cell
10.3 reduce `n_draws` to 50. Which distributions' histograms still look like their PMF/PDF?

## 📌 Key takeaways

- A **random variable** is a function $X: \Omega \to \mathbb{R}$; $P(X = x)$ adds the probabilities of the outcomes
  mapped to $x$. It is not a probability and not an event ("$X = 1$" is the event $\{HT, TH\}$). One $\Omega$ can carry
  many random variables.
- The **CDF** $F(x) = P(X \le x)$ exists for every random variable: non-decreasing, right-continuous, from 0 to 1;
  $P(a < X \le b) = F(b) - F(a)$; for discrete $X$ it is a staircase whose jumps are the PMF ($P(V \le 4) = P(V \le
  4.78) = 3/8$ for two 4-sided dice).
- **PMFs:** point mass $\delta_a$; **Bernoulli** $\alpha^k(1-\alpha)^{1-k}$ (mean $\alpha$, variance $\alpha(1-\alpha)$);
  **multinoulli** $\prod p_i^{\delta(x,i)}$; **binomial** $\binom nk \alpha^k(1-\alpha)^{n-k}$ (count sequences; mean
  $n\alpha$); **Poisson** $\lambda^k e^{-\lambda}/k!$ (rare events; limit of the binomial; horse kicks; read counts);
  **geometric** $(1-\alpha)^{k-1}\alpha$ (first success; memoryless).
- **PDFs:** probability = area, $\int f = 1$, $f = F'$, $f$ can exceed 1, $P(X = x) = 0$, $P([x, x+\Delta x]) \approx
  f(x)\Delta x$. **Uniform** $1/(b-a)$; **Gaussian** $\frac{1}{\sigma\sqrt{2\pi}}e^{-(x-\mu)^2/2\sigma^2}$ (CDF $\Phi$ via
  erf; z-scores; 68–95–99.7); **exponential** $\lambda e^{-\lambda x}$ (waiting times between Poisson events, mean
  $1/\lambda$).
- Biology: genotypes are Binomial(2, p); sequencing reads are binomial (Phred $Q = -10\log_{10}P(\text{error})$);
  technical RNA-seq counts are Poisson (variance = mean), biological ones over-dispersed; thresholds use $1 - F(t)$.

---
## ✅ Checkpoint

1. Toss a fair coin three times; $Y$ = number of heads. Give the PMF of $Y$ and $F_Y(1.5)$.
   <details><summary>Answer</summary>P(Y = 0) = 1/8, P(Y = 1) = 3/8, P(Y = 2) = 3/8, P(Y = 3) = 1/8. F_Y(1.5) = P(Y ≤ 1.5) = P(Y ≤ 1) = 1/8 + 3/8 = 1/2.</details>
2. Is "$X = 2$" a random variable, an event or a probability? And $X(\omega)$? And $P(X = 2)$?
   <details><summary>Answer</summary>"X = 2" is an event (the set of outcomes ω with X(ω) = 2); X(ω) is a number (the value of the function X at ω); P(X = 2) is a probability.</details>
3. A CDF jumps from 0.3 to 0.8 at $x = 4$ and is flat elsewhere between 3 and 5. What is $P(X = 4)$? $P(3 < X \le 5)$?
   <details><summary>Answer</summary>The jump size: P(X = 4) = 0.8 − 0.3 = 0.5. P(3 < X ≤ 5) = F(5) − F(3) = 0.8 − 0.3 = 0.5.</details>
4. Derive $E[X]$ and $V(X)$ for $X \sim$ Bernoulli($\alpha$).
   <details><summary>Answer</summary>E[X] = 1·α + 0·(1 − α) = α; E[X²] = 1²·α + 0²·(1 − α) = α; V(X) = E[X²] − (E[X])² = α − α² = α(1 − α).</details>
5. Explain where each factor of $\binom{n}{k}\alpha^k(1-\alpha)^{n-k}$ comes from, and compute $P(X = 2)$ for $X \sim$ Binomial(5, 0.2).
   <details><summary>Answer</summary>α^k(1 − α)^(n−k) is the probability of one particular sequence with k successes (independent trials, product rule); C(n, k) counts such sequences (which positions are successes); the sequences are disjoint, so their probabilities add. P(X = 2) = 10 · 0.04 · 0.512 = 0.2048.</details>
6. Bortkiewicz's horse kicks: with $\lambda = 0.61$, compute $P(X = 0)$ and the expected number of corps-years (out of 200) with no death.
   <details><summary>Answer</summary>P(X = 0) = e^(−0.61) ≈ 0.543; 200 × 0.543 ≈ 108.7 (observed: 109).</details>
7. Why is a Poisson distribution a good model for the number of reads of a gene in a technical replicate? What signature in the data would you check?
   <details><summary>Answer</summary>Each of millions of reads comes from the gene with a tiny probability, independently — many rare trials, the binomial-to-Poisson limit. Check that the variance across replicates is close to the mean (over-dispersion, variance > mean, indicates biological variability → negative binomial).</details>
8. Show that the geometric distribution is memoryless.
   <details><summary>Answer</summary>P(K > k) = (1 − α)^k (k failures in a row). P(K > m + j | K > m) = P(K > m + j)/P(K > m) = (1 − α)^(m+j)/(1 − α)^m = (1 − α)^j = P(K > j).</details>
9. $X \sim$ Uniform(0, 0.5). What is $f(0.25)$? Is it a probability? What is $P(X = 0.25)$?
   <details><summary>Answer</summary>f(0.25) = 1/(0.5 − 0) = 2 — a density, not a probability (it exceeds 1). P(X = 0.25) = 0.</details>
10. Cholesterol in a population is $N(190, 30^2)$ mg/dL (illustrative). What fraction exceeds 240 mg/dL?
    <details><summary>Answer</summary>z = (240 − 190)/30 = 1.667; P = 1 − Φ(1.667) ≈ 0.048 (about 5 %).</details>
11. Mutations occur along a genome at rate 2 per 10 kb (a Poisson process). What is the distribution of (a) the number of mutations in a 5-kb window and (b) the distance between neighbouring mutations? Give $P(\text{no mutation in 5 kb})$.
    <details><summary>Answer</summary>(a) Poisson with λ = 1 (2 per 10 kb × 5 kb); (b) exponential with rate 0.2 per kb (mean 5 kb). P(no mutation) = e^(−1) ≈ 0.368 — the same as P(gap > 5 kb) = e^(−0.2·5).</details>
12. Give the PMF/PDF/CDF that you would use for: the base at a position; the number of correct reads among 30; the time until a cell divides (rare event, constant rate); adult height.
    <details><summary>Answer</summary>Multinoulli (PMF); Binomial(30, α) (PMF); exponential (PDF); Gaussian (PDF). Every one also has a CDF.</details>

## 🧠 Questions & discussion

1. The binomial assumes independent trials with the same success probability. Name a sequencing situation in which read errors are *not* independent, and predict how the true probability of "majority wrong" compares with the binomial one.
2. Bortkiewicz's corps differed in size and in the number of horses. Why might the Poisson still fit, and when would a mixture of Poissons (over-dispersion) be needed?
3. Is human height really Gaussian? What about body weight, or blood concentrations of a hormone? What features of the data (skewness, a lower bound at 0) argue against the normal model?
4. A density can exceed 1 but a probability cannot. Which of the two does a histogram with `density=True` estimate, and why does the choice of bin width matter?
5. Memorylessness makes the geometric and exponential mathematically convenient. Is the time until a cell divides plausibly memoryless? What about the time until a radioactive atom decays?

## 🚀 Bonus — going further

- **The negative binomial.** Sample Poisson counts whose rate is itself Gamma-distributed (as `simulate_cell_counts(overdispersion=…)` does) and show that the variance is $\lambda + \varphi\lambda^2$. This is the count model of DESeq2 and edgeR ([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)).
- **Inverse-transform sampling.** If $U \sim$ Uniform(0, 1) then $F^{-1}(U)$ has CDF $F$. Use it to generate exponential samples from uniform ones, $-\ln(1 - U)/\lambda$, and check against `rng.exponential`.
- **Rare-event statistics in genomics.** Compute the probability that a 20-bp sequence (e.g. a CRISPR guide) occurs by chance at least once in a random 3-Gb genome (Poisson approximation with $\lambda = 2 \times 3 \times 10^9 / 4^{20}$, counting both strands).
- **More distributions.** Look up the gamma, beta, chi-square, t and log-normal distributions in `scipy.stats`; each reappears in Part F ([Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb), [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb), [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb)).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 18 — Random variables and their distributions — date:
Input: coin, dice and cross sample spaces; SIMULATED genotypes, reads, single-cell counts, mutation waits,
       expression values; Bortkiewicz's horse-kick table (typed in)
Output: PMFs and CDFs (exact fractions); Bernoulli/binomial/Poisson/geometric/uniform/Gaussian/exponential
        plots; Poisson fit to horse kicks; binomial → Poisson; tail probabilities; sampling histograms
Method: X(ω) as a Python function; grouping outcomes by value; counting sequences; numerical integration (quad)
Key equation: P(X = x) = Σ_{ω: X(ω)=x} p(ω);  F(x) = P(X ≤ x);  P(a ≤ X ≤ b) = ∫_a^b f(x) dx
Key parameters: α (success probability), n, λ (rate / mean count), μ and σ, threshold t
What changed when I modified λ / α / σ / the overdispersion:
One failure mode: (e.g., reading f(x) as a probability; NumPy's exponential scale vs rate)
Connection to 02-680 (which idea of random variables this illustrates):
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Sections 2.1–2.4 (random variables, distribution functions, discrete and continuous distributions).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapters 3 and 5 (random variables and special distributions).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — Sections 6.2 (discrete and continuous probabilities) and 6.5 (the Gaussian).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — the chapters on random variables and distributions.
- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*. Cambridge University Press — random variables.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- von Bortkiewicz L. (1898) *Das Gesetz der kleinen Zahlen* (The Law of Small Numbers). Teubner, Leipzig.
- Marioni J. C., Mason C. E., Mane S. M., Stephens M., Gilad Y. (2008) RNA-seq: an assessment of technical reproducibility and comparison with gene expression arrays. *Genome Research* 18:1509–1517.
- Li H. (2011) A statistical framework for SNP calling, mutation discovery, association mapping and population genetical parameter estimation from sequencing data. *Bioinformatics* 27:2987–2993.
- Ewing B., Green P. (1998) Base-calling of automated sequencer traces using phred. II. Error probabilities. *Genome Research* 8:186–194.
- Love M. I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biology* 15:550.

## ➡️ Next

[Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) — this module studied one random variable at a time. Biology is about relationships: a genotype and a disease,
two co-expressed genes, the first and second toss of a coin of unknown type. The next module puts several random
variables on one sample space — **joint**, **marginal** and **conditional** distributions, independence of random
variables, covariance and correlation, and the multivariate Gaussian whose 2-D density you glimpsed in Section 9.